<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_03_target_definition/stage_03a_target_investigation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_03a_target_investigation**


Vamos a reformular el Stage 03 partiendo exclusivamente del análisis empírico del mercado, y que el target sea una consecuencia de los datos, no un supuesto previo. El objetivo es que el MNQ nos “revele” cuál es una meta económicamente lógica, frecuente y operable.

## **Configuración del Entorno**


### 0.1. Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Importación de librerías


In [2]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

### 0.3. Definición de rutas

In [3]:
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [4]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/processed/mnq_intraday.parquet"))
OUT_PARQUET = Path(os.environ.get("OUT_PARQUET", "data/processed/mnq_intraday_labeled.parquet"))
OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/target_definitio_summary.json"))

In [5]:
# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
OUT_PARQUET = DRIVE_DIR / OUT_PARQUET
OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

### 0.4. Carga de dataset `intraday_mnq`


In [6]:
def load_mnq_parquet():
    os.path.exists(IN_PARQUET)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(IN_PARQUET)
    return mnq_parquet

In [7]:
def add_column_date(df):
    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [8]:
def info_dataset(df):
  print("Información del dataset:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}")

In [9]:
mnq_intraday = load_mnq_parquet()
mnq_intraday = add_column_date(mnq_intraday)
mnq_intraday.head()

Archivo encontrado en disco. Cargando dataset local...


,date,open,high,low,close,volume
datetime,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3


In [10]:
info_dataset(mnq_intraday)

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 571
	Hora diaria de inicio 06:30
	Hora diaria de final 16:00
	Zona horaria: America/New_York


# **1. Investigación y definición empírica del objetivo de predicción**

## 1.1. Marco teórico

En problemas de predicción financiera intradía, la definición del *target* es una de las decisiones metodológicas más críticas, porque determina simultáneamente:

- la **viabilidad estadística** del aprendizaje (estabilidad, escala, ruido, estacionariedad), y  
- la **utilidad económica** del modelo (interpretabilidad, PnL, ejecución real).

En este trabajo se consideran dos formulaciones habituales del objetivo de predicción para un horizonte \( h \):

1. **Delta en puntos (movimiento absoluto del precio)**  

$$
\Delta P_{t,h} = P_{t+h} - P_t
$$

2. **Retorno (movimiento relativo del precio)**  

$$
r_{t,h} = \frac{P_{t+h} - P_t}{P_t}
$$

(opcionalmente también se puede evaluar el log-return):

$$
\ell_{t,h} = \ln\left(\frac{P_{t+h}}{P_t}\right)
$$

Ambas definiciones describen el mismo fenómeno (movimiento futuro del precio), pero **imponen propiedades estadísticas distintas** y conducen a decisiones diferentes de modelado y evaluación.



## 1.2. Limitaciones de definir targets de forma exógena (umbrales fijados a priori)

En enfoques tradicionales, el objetivo suele definirse fijando umbrales de movimiento (por ejemplo ±25 o ±62.5 puntos, o retornos mínimos) basados en metas deseadas o heurísticas. Este enfoque presenta varias limitaciones estructurales:

**(a) Desacople con la dinámica real del mercado**  
Umbrales arbitrarios pueden corresponder a eventos poco frecuentes o concentrados en ventanas horarias específicas, generando datasets altamente desbalanceados y modelos con baja capacidad de generalización.

**(b) Riesgo de optimización ilusoria**  
Un modelo puede mostrar métricas estadísticas aceptables sobre un target mal definido, pero resultar económicamente inviable al ser aplicado en condiciones reales de trading.

**(c) Heterogeneidad intradía y microestructura**  
En horizontes cortos, la distribución de los movimientos de precio está fuertemente condicionada por la microestructura del mercado, la volatilidad intradía y el régimen horario, lo que invalida supuestos homogéneos sobre la magnitud de los movimientos futuros.



## 1.3. Deltas vs retornos: trade-off estadístico y operativo



El **delta en puntos** es una magnitud directamente operativa y fácilmente interpretable como PnL en puntos del instrumento. Sin embargo:

- su escala puede depender del **nivel de precio** (cambios de régimen entre distintos períodos históricos),  
- puede ser más sensible a heterocedasticidad intradía y a cambios en la volatilidad.

Los **retornos**, en cambio, normalizan el movimiento por el nivel de precio y suelen:

- ser más comparables en el tiempo y entre distintos activos,  
- favorecer una mayor estabilidad estadística y, en muchos contextos, un aprendizaje más robusto.

No obstante, desde el punto de vista operativo:

- el trader ejecuta en puntos, por lo que una señal basada en retornos debe convertirse nuevamente a delta de puntos:

$$
\widehat{\Delta P}_{t,h} \approx P_t \cdot \hat r_{t,h}
$$

Esta conversión puede introducir errores adicionales dependientes del nivel de precio $ P_t $.

Por este motivo, **no se asume a priori** que una formulación sea superior a la otra. La elección del target se aborda como un problema empírico, evaluando cuál de las dos alternativas logra mayor coherencia estadístico–económica en el dataset intradía del MNQ.



## 1.4. Enfoque empírico orientado al mercado

Se adopta un enfoque empírico en el cual el target **emerge directamente de las distribuciones observadas** en los datos históricos:

- se calculan $ \Delta P_{t,h} $ y $ r_{t,h} $ para horizontes definidos,  
- se analiza su comportamiento por jornada y por régimen horario,  
- se evalúa su estabilidad temporal y su relación con las variables explicativas (*features*).

Este enfoque busca que la definición del objetivo de predicción sea una consecuencia natural del dataset y de la dinámica real del mercado, y no una decisión impuesta externamente.



## 1.5. Principio de coherencia estadístico–económica

El criterio central que guía este estudio es el de **coherencia estadístico–económica**:

Un objetivo de predicción es válido si y solo si representa un movimiento que:

1. ocurre con **frecuencia suficiente** en los datos históricos, y  
2. es **económicamente significativo** en términos de PnL neto y ejecución real.

Bajo este principio, la comparación entre “retornos vs deltas” no se resuelve por preferencia teórica, sino por evidencia empírica, considerando:

- estabilidad temporal,  
- comportamiento de colas y eventos extremos,  
- sensibilidad al nivel de precio,  
- desempeño de modelos baseline comparables.

## 1.6. Pregunta metodológica correcta

Este marco teórico desplaza el foco desde la pregunta:

> “¿Puede el modelo predecir este objetivo?”

hacia una pregunta metodológicamente más adecuada:

> “¿Qué objetivo tiene sentido predecir dadas las propiedades empíricas del mercado y del dataset?”

Solo después de responder esta última resulta legítimo avanzar hacia la definición de etiquetas, selección de métricas, entrenamiento de modelos y evaluación mediante backtesting.

#**2. Dataset de partida para la investigación de targets**


El análisis se realiza sobre el dataset intradía del contrato MNQ, estructurado a nivel de minuto, con precios OHLCV y sin etiquetas de predicción predefinidas. Cada fila representa un instante temporal dentro de una sesión de trading, preservando la estructura intradía y evitando cruces entre jornadas.

A partir de este dataset se construirán dos targets alternativos, $\Delta P_{t,h} $ y $ r_{t,h} $, para distintos horizontes $ h $ (por ejemplo, 60 y 90 minutos), y se estudiará su comportamiento empírico como paso previo a la elección del objetivo de predicción final.

In [11]:
mnq_intraday

,date,open,high,low,close,volume
datetime,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3
...,...,...,...,...,...,...
2025-06-13 15:56:00-04:00,2025-06-13,21624.50,21635.00,21613.50,21617.50,3251
2025-06-13 15:57:00-04:00,2025-06-13,21616.50,21635.25,21615.75,21623.75,2201
2025-06-13 15:58:00-04:00,2025-06-13,21623.25,21632.75,21616.50,21621.75,1859


## 2.0. Utilidades

In [12]:
import numpy as np
import pandas as pd

def compute_targets_delta_and_returns(
    df: pd.DataFrame,
    *,
    close_col: str = "close",
    date_col: str = "date",
    horizons: tuple[int, ...] = (60, 90),
) -> pd.DataFrame:
    """
    Fórmulas:
      ΔP_{t,h} = P_{t+h} - P_t
      r_{t,h}  = (P_{t+h} - P_t) / P_t
      ℓ_{t,h}  = ln(P_{t+h} / P_t)

    Sin cruzar días (shift por date). Índice datetime intacto.
    """
    out = df.copy()

    for h in horizons:
        P_t = out[close_col]
        P_th = out.groupby(date_col, group_keys=False)[close_col].shift(-h)

        out[f"delta_{h}"] = P_th - P_t
        out[f"ret_{h}"]   = (P_th - P_t) / P_t
        out[f"lret_{h}"]  = np.log(P_th / P_t)

    return out


In [13]:
import numpy as np
import pandas as pd

def validate_targets_no_cross_day(
    df: pd.DataFrame,
    *,
    close_col: str = "close",
    date_col: str = "date",
    horizons: tuple[int, ...] = (60, 90),
    rtol: float = 1e-10,
    atol: float = 1e-12,
) -> None:
    """
    Comprueba:
      1) ret_h == (P_{t+h}-P_t)/P_t  (y equivalencia con P_{t+h}/P_t - 1)
      2) lret_h == ln(P_{t+h}/P_t)
      3) No cruza días: donde exista P_{t+h}, su 'date' coincide con date actual
      4) NaNs por día: en cada día, los últimos h registros tienen NaN (exactamente h)

    Lanza AssertionError si algo falla.
    """
    if close_col not in df.columns or date_col not in df.columns:
        raise ValueError(f"Faltan columnas: requiere '{close_col}' y '{date_col}'")

    for h in horizons:
        for col in (f"delta_{h}", f"ret_{h}", f"lret_{h}"):
            if col not in df.columns:
                raise ValueError(f"Falta la columna requerida: '{col}'")

        P_t = df[close_col]
        P_th = df.groupby(date_col, group_keys=False)[close_col].shift(-h)

        # -----------------------------
        # 1) Verificación fórmulas
        # -----------------------------
        expected_delta = P_th - P_t
        expected_ret   = (P_th - P_t) / P_t
        expected_ret2  = (P_th / P_t) - 1.0
        expected_lret  = np.log(P_th / P_t)

        # comparar solo donde hay datos (no NaN)
        m = P_th.notna() & P_t.notna()

        assert np.allclose(df.loc[m, f"delta_{h}"].to_numpy(),
                           expected_delta.loc[m].to_numpy(),
                           rtol=rtol, atol=atol), f"delta_{h} no coincide con P_th - P_t"

        assert np.allclose(df.loc[m, f"ret_{h}"].to_numpy(),
                           expected_ret.loc[m].to_numpy(),
                           rtol=rtol, atol=atol), f"ret_{h} no coincide con (P_th - P_t)/P_t"

        # equivalencia algebraica
        assert np.allclose(expected_ret.loc[m].to_numpy(),
                           expected_ret2.loc[m].to_numpy(),
                           rtol=rtol, atol=atol), f"ret_{h} no es equivalente a P_th/P_t - 1"

        assert np.allclose(df.loc[m, f"lret_{h}"].to_numpy(),
                           expected_lret.loc[m].to_numpy(),
                           rtol=rtol, atol=atol), f"lret_{h} no coincide con ln(P_th/P_t)"

        # -----------------------------
        # 2) No cruza días (verifica date de t+h)
        # -----------------------------
        date_t  = df[date_col]
        date_th = df.groupby(date_col, group_keys=False)[date_col].shift(-h)

        # donde exista t+h, debe ser el mismo date
        m_date = date_th.notna()
        assert (date_th.loc[m_date] == date_t.loc[m_date]).all(), (
            f"Cruce de día detectado en h={h}: date(t+h) != date(t)"
        )

        # -----------------------------
        # 3) NaNs solo en últimos h de cada día (exactamente h)
        # -----------------------------
        nan_counts = df[f"ret_{h}"].isna().groupby(df[date_col]).sum()
        # en días con longitud >= h, esperamos exactamente h NaNs (los últimos h)
        sizes = df.groupby(date_col).size()
        check_days = sizes[sizes >= h].index
        assert (nan_counts.loc[check_days] == h).all(), (
            f"Conteo de NaNs por día incorrecto para h={h}. "
            f"Se esperaba h NaNs por día (en días con >=h filas)."
        )

    print("Validación OK: fórmulas, no cruce de días y NaNs por jornada/horizonte.")


# --- Ejecución ---
# validate_targets_no_cross_day(mnq_intraday_targets, close_col="close", date_col="date", horizons=(60, 90))


## 2.1. Calculo de targets

In [14]:
mnq_intraday_targets = compute_targets_delta_and_returns(
    mnq_intraday,
    close_col="close",
    date_col="date",
    horizons=(60, 90),
)

validate_targets_no_cross_day(mnq_intraday_targets)

Validación OK: fórmulas, no cruce de días y NaNs por jornada/horizonte.


# **3. Metodología de análisis de los targets**

Sobre los targets calculados (`delta_60/90`, `ret_60/90`, `lret_60/90`) se realizará un análisis estructurado en **tres capas**, ordenadas de menor a mayor complejidad y costo computacional. El objetivo es decidir, con evidencia empírica, cuál formulación resulta más adecuada como objetivo de predicción.


## 3.1. Diagnóstico descriptivo y de riesgo



Para cada horizonte (60 y 90 minutos) y para cada tipo de target (delta, retorno y log-retorno) se analizará:

- **Distribución básica**  
  Media, mediana, desviación estándar, rango intercuartílico (IQR) y percentiles  
  (1, 5, 50, 95 y 99).

- **Asimetría y colas**  
  Skewness, kurtosis y proporción de observaciones extremas.

- **Simetría direccional**  
  Porcentaje de valores positivos, negativos y cercanos a cero.

**Objetivo:**  
Entender la magnitud típica de los movimientos, la presencia de colas pesadas y si el target está dominado por ruido o por eventos extremos poco frecuentes.


### 3.1.1. Tabla de estadísticos básicos + percentiles + colas + simetría

In [15]:
import numpy as np
import pandas as pd

def summarize_targets_descriptive(
    df: pd.DataFrame,
    *,
    targets: list[str],
    near_zero_mode: str = "delta",   # "delta" o "return"
    near_zero_value: float = 0.0,    # si es 0, se define automáticamente
) -> pd.DataFrame:
    """
    Genera un resumen descriptivo para una lista de columnas target.

    Incluye:
      - n, n_nan
      - media, mediana, std
      - IQR
      - percentiles (1, 5, 50, 95, 99)
      - skewness y kurtosis (Fisher, exceso de curtosis)
      - simetría direccional: %pos, %neg, %near_zero
      - proporción de extremos: |x| >= p99 (extremos superiores en magnitud)

    near_zero:
      - Para deltas: se sugiere umbral en puntos, por ejemplo 1.0 pt.
      - Para retornos: se sugiere umbral relativo, por ejemplo 0.0005 (5 bps).
      - Si near_zero_value=0, se define por default según near_zero_mode.
    """
    rows = []

    # Definir umbral "cerca de cero" si el usuario no lo setea
    # (esto solo es un default inicial; luego podemos ajustarlo)
    if near_zero_value == 0.0:
        if near_zero_mode == "delta":
            near_zero_value = 1.0        # 1 punto como "casi cero" (ajustable)
        elif near_zero_mode == "return":
            near_zero_value = 0.0005     # 5 bps como "casi cero" (ajustable)
        else:
            raise ValueError("near_zero_mode debe ser 'delta' o 'return'")

    for col in targets:
        s = df[col].astype(float)

        # Datos válidos (sin NaN)
        x = s.dropna()

        # Conteos básicos
        n = int(x.shape[0])
        n_nan = int(s.isna().sum())

        if n == 0:
            # Si no hay datos válidos, devolvemos fila vacía con NaNs
            rows.append({"target": col, "n": 0, "n_nan": n_nan})
            continue

        # Estadísticos básicos
        mean = float(x.mean())
        median = float(x.median())
        std = float(x.std(ddof=1))

        # Percentiles
        p01 = float(x.quantile(0.01))
        p05 = float(x.quantile(0.05))
        p50 = float(x.quantile(0.50))
        p95 = float(x.quantile(0.95))
        p99 = float(x.quantile(0.99))

        # IQR (p75 - p25)
        p25 = float(x.quantile(0.25))
        p75 = float(x.quantile(0.75))
        iqr = float(p75 - p25)

        # Asimetría y colas
        skew = float(x.skew())
        # Kurtosis en pandas default es "Fisher" (excess kurtosis). Normal => 0.
        kurt_excess = float(x.kurt())

        # Simetría direccional
        pct_pos = float((x > 0).mean())
        pct_neg = float((x < 0).mean())
        pct_near_zero = float((x.abs() <= near_zero_value).mean())

        # Proporción de extremos por magnitud:
        # Definimos extremo como |x| >= p99(|x|)
        abs_x = x.abs()
        abs_p99 = float(abs_x.quantile(0.99))
        pct_extreme_abs_p99 = float((abs_x >= abs_p99).mean())

        rows.append({
            "target": col,
            "n": n,
            "n_nan": n_nan,
            "mean": mean,
            "median": median,
            "std": std,
            "iqr": iqr,
            "p01": p01,
            "p05": p05,
            "p50": p50,
            "p95": p95,
            "p99": p99,
            "skew": skew,
            "kurt_excess": kurt_excess,
            "pct_pos": pct_pos,
            "pct_neg": pct_neg,
            "pct_near_zero": pct_near_zero,
            "abs_p99": abs_p99,
            "pct_extreme_abs_p99": pct_extreme_abs_p99,
            "near_zero_value_used": near_zero_value,
        })

    out = pd.DataFrame(rows)

    # Ordenar por target para lectura
    out = out.sort_values("target").reset_index(drop=True)

    return out

In [16]:
# ------------------------------------------------------------
# Ejecutar el resumen para H=60 y H=90 (delta/ret/lret)
# ------------------------------------------------------------

targets_60 = ["delta_60", "ret_60", "lret_60"]
targets_90 = ["delta_90", "ret_90", "lret_90"]

# Para deltas (umbral cerca de cero en puntos)
summary_60_delta_mode = summarize_targets_descriptive(
    mnq_intraday_targets,
    targets=targets_60,
    near_zero_mode="delta",
    near_zero_value=1.0,   # ajuste fino después si desea
)

summary_90_delta_mode = summarize_targets_descriptive(
    mnq_intraday_targets,
    targets=targets_90,
    near_zero_mode="delta",
    near_zero_value=1.0,
)

In [17]:
print('\nsummary_60_delta_mode:\n')
display(summary_60_delta_mode)
print('\nsummary_90_delta_mode:\n')
display(summary_90_delta_mode)


summary_60_delta_mode:



,target,n,n_nan,mean,median,std,iqr,p01,p05,p50,p95,p99,skew,kurt_excess,pct_pos,pct_neg,pct_near_zero,abs_p99,pct_extreme_abs_p99,near_zero_value_used
0,delta_60,665833,78180,0.481132,2.750000,59.742892,53.500000,-174.750000,-95.000000,2.750000,86.750000,155.750000,0.457314,21.675740,0.528272,0.468883,0.024856,204.250000,0.010007,1.0
1,lret_60,665833,78180,0.000046,0.000183,0.004270,0.003709,-0.012435,-0.006737,0.000183,0.006141,0.011478,0.281722,14.432596,0.528272,0.468883,1.000000,0.014563,0.010001,1.0
2,ret_60,665833,78180,0.000055,0.000183,0.004273,0.003709,-0.012358,-0.006714,0.000183,0.006160,0.011544,0.388694,15.373974,0.528272,0.468883,1.000000,0.014560,0.010001,1.0



summary_90_delta_mode:



,target,n,n_nan,mean,median,std,iqr,p01,p05,p50,p95,p99,skew,kurt_excess,pct_pos,pct_neg,pct_near_zero,abs_p99,pct_extreme_abs_p99,near_zero_value_used
0,delta_90,626743,117270,0.826449,3.500000,74.157814,68.000000,-214.750000,-118.750000,3.500000,108.750000,190.750000,0.581824,20.137614,0.530584,0.467144,0.020046,249.500000,0.010025,1.0
1,lret_90,626743,117270,0.000076,0.000244,0.005267,0.004731,-0.015035,-0.008441,0.000244,0.007738,0.013842,0.360458,12.036700,0.530584,0.467144,1.000000,0.017612,0.010001,1.0
2,ret_90,626743,117270,0.000090,0.000244,0.005273,0.004732,-0.014923,-0.008405,0.000244,0.007768,0.013938,0.473223,13.043359,0.530584,0.467144,1.000000,0.017571,0.010001,1.0


#### 3.1.1.a Interpretación de las variables del resumen descriptivo

Las tablas `summary_60_delta_mode` y `summary_90_delta_mode` contienen estadísticas descriptivas y de riesgo calculadas para cada target. A continuación se explica **qué mide cada variable y cómo debe interpretarse** en el contexto de la investigación de targets.

---

**Identificación y tamaño de muestra**

- **target**  
  Nombre del objetivo analizado (por ejemplo `delta_60`, `ret_60`, `lret_60`).

- **n**  
  Cantidad de observaciones válidas (no NaN) disponibles para ese target.  
  Refleja el tamaño efectivo de la muestra usada para el análisis.

- **n_nan**  
  Cantidad de valores faltantes.  
  En este caso, corresponde principalmente a los últimos minutos de cada jornada donde no existe \( t+h \).

---

**Tendencia central y dispersión**

- **mean**  
  Media aritmética del target.  
  En intradía suele ser cercana a cero; valores positivos indican sesgo alcista promedio.

- **median**  
  Mediana de la distribución.  
  Es más robusta que la media frente a eventos extremos; útil para identificar el “movimiento típico”.

- **std**  
  Desviación estándar.  
  Mide la dispersión global del target; valores altos indican alta volatilidad del movimiento futuro.

- **iqr** (Interquartile Range)  
  Diferencia entre el percentil 75 y 25.  
  Representa la dispersión “central” del 50% de los datos, menos sensible a colas extremas que `std`.

---

**Percentiles (estructura de la distribución)**

- **p01, p05**  
  Percentiles inferiores (1% y 5%).  
  Describen la magnitud de movimientos negativos extremos.

- **p50**  
  Percentil 50 (mediana).  
  Coincide con `median`.

- **p95, p99**  
  Percentiles superiores (95% y 99%).  
  Describen la magnitud de movimientos positivos extremos.

Estos percentiles permiten evaluar asimetrías y el tamaño típico de colas en ambos sentidos.

---

**Forma de la distribución**

- **skew**  
  Asimetría de la distribución.  
  - Valor positivo: cola derecha más pesada (eventos positivos más extremos).  
  - Valor negativo: cola izquierda más pesada.

- **kurt_excess**  
  Curtosis en exceso (Fisher).  
  - `0`: distribución normal  
  - `> 0`: colas pesadas (eventos extremos más frecuentes que en una normal)

En intradía, valores altos son esperables y relevantes para la evaluación de riesgo.

---

**Simetría direccional**

- **pct_pos**  
  Proporción de observaciones positivas.  
  Idealmente cercana a 0.5 en mercados sin sesgo direccional fuerte.

- **pct_neg**  
  Proporción de observaciones negativas.  
  Complementaria a `pct_pos`.

Estas métricas permiten verificar si el target está fuertemente desbalanceado en dirección.

---

**Zona de “ruido” o movimientos pequeños**

- **pct_near_zero**  
  Proporción de observaciones cuyo valor absoluto es menor o igual al umbral definido como “cerca de cero”.

  - Para `delta_*`: el umbral se expresa en puntos.
  - Para `ret_*` y `lret_*`: el umbral se expresa en retorno (relativo).

  Valores altos indican que el target está dominado por movimientos pequeños, potencialmente difíciles de predecir o poco explotables económicamente.

- **near_zero_value_used**  
  Umbral utilizado para definir “cerca de cero” en ese resumen.  
  En estas tablas se fijó en `1.0` como valor inicial común.

---

**Eventos extremos por magnitud**

- **abs_p99**  
  Percentil 99 de \(|target|\).  
  Representa el tamaño típico de un evento extremo en magnitud absoluta.

- **pct_extreme_abs_p99**  
  Proporción de observaciones con \(|target| \geq \text{abs\_p99}\).  
  Por definición, suele estar cerca del 1%, y sirve como chequeo de consistencia y referencia para colas.

---

**Lectura general**

Estas variables permiten evaluar simultáneamente:

- la **escala** del target (puntos vs retornos),
- la **estabilidad estadística**,
- la **importancia relativa de eventos extremos**,
- y la **proporción de ruido intradía**.

Esta información es la base para decidir si un target es adecuado desde el punto de vista estadístico antes de evaluar su predecibilidad con modelos.


#### 3.1.1.b Observaciones y conclusiones – Diagnóstico descriptivo (H = 60 y 90)

**Observaciones generales**
- En ambos horizontes, **deltas y retornos presentan ligera asimetría positiva**, con ~53% de movimientos positivos.
- Las distribuciones muestran **colas pesadas** (kurtosis elevada), típico del comportamiento intradía.
- El tamaño muestral es grande y comparable entre targets, lo que hace confiables las estadísticas.

---

**Horizonte H = 60 minutos**

- **delta_60**
  - Movimiento típico bajo (mediana ≈ 2.75 pts) frente a una dispersión alta (std ≈ 60 pts).
  - Colas muy pesadas (kurtosis ≈ 21), con eventos extremos de hasta ±200 pts.
  - Poca zona “casi cero” (~2.5%), lo que indica movimientos medibles con frecuencia.

- **ret_60 / lret_60**
  - Mediana muy cercana a cero, con dispersión estable y menor sensibilidad a colas extremas.
  - Kurtosis elevada pero menor que en delta → distribución más controlada.
  - Con el umbral usado (1.0), prácticamente **todo el target queda clasificado como “near zero”**, lo que indica que este umbral no es informativo para retornos.

---

**Horizonte H = 90 minutos**

- **delta_90**
  - Aumenta la magnitud típica (mediana ≈ 3.5 pts) y la dispersión (std ≈ 74 pts).
  - Colas extremas más amplias (p99 ≈ ±250 pts).
  - Sigue existiendo poca masa cerca de cero (~2%), manteniendo relevancia operativa.

- **ret_90 / lret_90**
  - Mayor dispersión que en H=60, coherente con el horizonte más largo.
  - Kurtosis menor que en delta, indicando colas más “suavizadas”.
  - Nuevamente, el umbral de “near zero” no discrimina movimientos pequeños.

---

**Comparación delta vs retorno**

- **Delta en puntos** refleja bien la magnitud económica real, pero exhibe colas más pesadas y mayor heterocedasticidad.
- **Retornos y log-retornos** presentan distribuciones más estables y comparables entre horizontes, pero requieren redefinir el umbral de “near zero” para ser informativos.
- A mayor horizonte, ambos targets aumentan su dispersión, pero el **crecimiento es más pronunciado en delta**.

---

**Conclusión preliminar**

- El **delta en puntos** es claramente interpretable y económicamente significativo, pero estadísticamente más riesgoso.
- Los **retornos** ofrecen mayor estabilidad estadística, a costa de perder interpretabilidad directa en puntos.
- Antes de decidir el target final, es necesario ajustar el análisis de “near zero” para retornos y avanzar a la evaluación de estabilidad intradía y predecibilidad.


### 3.1.2. Ajuste del umbral “near zero” por tipo de target

#### 3.1.2.a. Motivación



- En el punto 3.1.1 se observó que:

  - Para **deltas**, un umbral fijo expresado en puntos (por ejemplo ±1 pt) discrimina adecuadamente los movimientos pequeños.
  - Para **retornos y log-retornos**, utilizar el mismo umbral numérico (1.0) **no tiene sentido económico**, ya que prácticamente todas las observaciones quedan clasificadas como “near zero”.

Esto se debe a que:

- los deltas están expresados en **puntos absolutos de precio**,  
- los retornos están expresados en **unidades relativas**, típicamente del orden de basis points (bps).

Por lo tanto, el concepto de *movimiento pequeño* debe definirse **en la escala correcta de cada target**, para que el análisis sea informativo y comparable.

---

**Definición de umbrales a evaluar**

Se evaluarán **múltiples umbrales** con el objetivo de analizar la sensibilidad del criterio *near zero*.

Para deltas (en puntos):
- ±1.0 pt  
- ±2.0 pts  
- ±5.0 pts  

Para retornos y log-retornos (en basis points):
- 1 bp  = 0.0001  
- 5 bps = 0.0005  
- 10 bps = 0.0010  

Esta evaluación permitirá responder preguntas como:
- ¿Qué porcentaje del tiempo el mercado se mueve menos que un umbral económicamente relevante?
- ¿Qué proporción del target está dominada por ruido de baja magnitud?

---

**Métricas recalculadas**

Para cada target y para cada umbral considerado se recalcula únicamente:

- **pct_near_zero**  
  Proporción de observaciones que cumplen:
  
 $$
  |target| \le \text{umbral}
  $$

No se recalculan otras métricas descriptivas en este punto, ya que fueron analizadas previamente en la sección 3.1.1.


#### 3.1.2.b. Código: cálculo de % near zero por múltiples umbrales

In [18]:
import pandas as pd
import numpy as np

def compute_near_zero_by_thresholds(
    df: pd.DataFrame,
    *,
    targets: list[str],
    thresholds: list[float],
) -> pd.DataFrame:
    """
    Calcula el porcentaje de observaciones 'near zero' para distintos umbrales.

    Para cada target y cada threshold:
      pct_near_zero = mean(|target| <= threshold)

    Retorna un DataFrame largo (tidy):
      target | threshold | pct_near_zero | n
    """
    rows = []

    for col in targets:
        x = df[col].dropna().astype(float)
        n = int(x.shape[0])

        if n == 0:
            continue

        abs_x = x.abs()

        for thr in thresholds:
            pct_near_zero = float((abs_x <= thr).mean())

            rows.append({
                "target": col,
                "threshold": thr,
                "pct_near_zero": pct_near_zero,
                "n": n,
            })

    return pd.DataFrame(rows)


In [19]:
# Deltas en puntos
delta_thresholds_pts = [1.0, 2.0, 5.0]
targets_delta_60 = ["delta_60"]

near_zero_delta_60 = compute_near_zero_by_thresholds(
    mnq_intraday_targets,
    targets=targets_delta_60,
    thresholds=delta_thresholds_pts,
)

# Retornos y log-retornos en bps
return_thresholds = [0.0001, 0.0005, 0.0010]  # 1, 5, 10 bps
targets_ret_60 = ["ret_60", "lret_60"]

near_zero_ret_60 = compute_near_zero_by_thresholds(
    mnq_intraday_targets,
    targets=targets_ret_60,
    thresholds=return_thresholds,
)



In [20]:
# Deltas en puntos
targets_delta_90 = ["delta_90"]

near_zero_delta_90 = compute_near_zero_by_thresholds(
    mnq_intraday_targets,
    targets=targets_delta_90,
    thresholds=delta_thresholds_pts,
)

# Retornos y log-retornos
targets_ret_90 = ["ret_90", "lret_90"]

near_zero_ret_90 = compute_near_zero_by_thresholds(
    mnq_intraday_targets,
    targets=targets_ret_90,
    thresholds=return_thresholds,
)



Observar:

- Si con umbrales pequeños (±1 pt o 1 bp) el `pct_near_zero` es muy alto → target dominado por ruido.
- Si al aumentar el umbral el porcentaje cae lentamente → movimientos distribuidos, target informativo.
- Comparar delta vs retorno al mismo “significado económico” (ej. 5 pts vs 5 bps).

#### 3.1.2.d. Código: consolidación de resultados

In [21]:
import pandas as pd

def consolidate_near_zero_tables(
    *,
    near_zero_delta_60: pd.DataFrame,
    near_zero_ret_60: pd.DataFrame,
    near_zero_delta_90: pd.DataFrame,
    near_zero_ret_90: pd.DataFrame,
) -> pd.DataFrame:
    """
    Consolida las tablas de near-zero (delta y retorno, H=60 y H=90)
    en una única tabla larga con columnas:

      horizon | target | target_type | threshold | pct_near_zero | n

    No recalcula métricas, solo agrega metadatos y concatena.
    """

    # --------
    # H = 60
    # --------
    delta_60 = near_zero_delta_60.copy()
    delta_60["horizon"] = 60
    delta_60["target_type"] = "delta"

    ret_60 = near_zero_ret_60.copy()
    ret_60["horizon"] = 60
    ret_60["target_type"] = "return"

    # --------
    # H = 90
    # --------
    delta_90 = near_zero_delta_90.copy()
    delta_90["horizon"] = 90
    delta_90["target_type"] = "delta"

    ret_90 = near_zero_ret_90.copy()
    ret_90["horizon"] = 90
    ret_90["target_type"] = "return"

    # --------
    # Concatenar todo
    # --------
    out = pd.concat(
        [delta_60, ret_60, delta_90, ret_90],
        axis=0,
        ignore_index=True,
    )

    # Reordenar columnas para lectura
    out = out[
        ["horizon", "target", "target_type", "threshold", "pct_near_zero", "n"]
    ].sort_values(
        ["horizon", "target_type", "target", "threshold"]
    ).reset_index(drop=True)

    return out


In [22]:
near_zero_all = consolidate_near_zero_tables(
    near_zero_delta_60=near_zero_delta_60,
    near_zero_ret_60=near_zero_ret_60,
    near_zero_delta_90=near_zero_delta_90,
    near_zero_ret_90=near_zero_ret_90,
)

display(near_zero_all)


,horizon,target,target_type,threshold,pct_near_zero,n
0,60,delta_60,delta,1.0000,0.024856,665833
1,60,delta_60,delta,2.0000,0.047085,665833
2,60,delta_60,delta,5.0000,0.113111,665833
3,60,lret_60,return,0.0001,0.032055,665833
4,60,lret_60,return,0.0005,0.158176,665833
5,60,lret_60,return,0.0010,0.301230,665833
6,60,ret_60,return,0.0001,0.032056,665833
7,60,ret_60,return,0.0005,0.158184,665833
8,60,ret_60,return,0.0010,0.301209,665833
9,90,delta_90,delta,1.0000,0.020046,626743


#### 3.1.2.c. Conclusiones – Análisis de “near zero” por umbral y horizonte


- Para **deltas**, los movimientos muy pequeños son poco frecuentes:  
  incluso con ±1 punto, solo ~2-2.5% de las observaciones se consideran “near zero”.  
  Esto indica que el mercado intradía del MNQ rara vez permanece completamente plano.

- Al aumentar el umbral en **deltas** (±2 y ±5 pts), la proporción de movimientos pequeños crece de forma gradual, pero sigue siendo moderada (`<12%` en H=60 y `<9%` en H=90 para ±5 pts).  
  El delta conserva contenido económico relevante incluso con umbrales relativamente amplios.

- Para **retornos y log-retornos**, el comportamiento depende fuertemente del umbral elegido:  
  con 1 bp, la proporción “near zero” es comparable a la de ±1 pt en deltas (~2.5-3%),  
  pero con 5-10 bps el porcentaje de movimientos pequeños crece rápidamente (15-30%).

- **Retorno y log-retorno muestran resultados prácticamente idénticos**, confirmando que, a estos horizontes, la elección entre ambos no cambia el diagnóstico de ruido.

- Al pasar de **H=60 a H=90**, la proporción de “near zero” disminuye levemente en todos los casos, reflejando que horizontes más largos capturan movimientos más amplios y menos ruido relativo.

- En términos de **ruido vs señal**, los deltas son menos sensibles al umbral y preservan señal económica con mayor consistencia, mientras que los retornos requieren una selección cuidadosa del umbral para no quedar dominados por movimientos pequeños.

**Conclusión preliminar:**  
Ambas formulaciones son válidas, pero los **deltas en puntos** muestran una relación más directa y estable entre umbral económico y contenido informativo, mientras que los **retornos** ofrecen mayor flexibilidad estadística a costa de una mayor sensibilidad a la definición de “near zero”.

### 3.1.3. Conclusiones de la sección 3.1

El análisis realizado **no descarta el uso de retornos como target de predicción**. Tanto los **deltas en puntos** como los **retornos** resultan formulaciones válidas desde el punto de vista empírico.

Los **deltas en puntos** presentan una ventaja en términos de **interpretación económica directa**: los umbrales tienen un significado claro y estable, y la relación entre magnitud del movimiento y contenido informativo es inmediata.

Los **retornos**, en cambio, ofrecen **mayor flexibilidad y estabilidad estadística**, pero requieren un **mayor cuidado en la definición de la escala y del umbral de “movimiento pequeño”**. Si este umbral no se elige adecuadamente, el target puede quedar dominado por ruido; cuando sí se define correctamente, los retornos pueden ser tan informativos como los deltas.

En síntesis, la elección entre deltas y retornos no es excluyente ni teórica, sino **una decisión empírica y de diseño**, que debe balancear interpretabilidad económica y robustez estadística. Esta conclusión habilita avanzar al análisis de estabilidad intradía y predecibilidad sin descartar ninguna de las dos formulaciones.


## 3.2. Estabilidad y regímenes intradía

Se evaluará cómo varía el comportamiento del target a lo largo del día y entre jornadas:

- **Por hora / minuto del día**  
  Comparación de dispersión y sesgo (por ejemplo, apertura vs cierre).

- **Por día**  
  Estadísticos diarios y visualizaciones tipo boxplot para analizar mediana y volatilidad diaria.

- **Relación con el nivel de precio** (clave para comparar delta vs retorno):  
  - correlación entre $|\Delta P|$ y `close`  
  - correlación entre $|r|$ y `close`  

  Si $|\Delta P|$ crece sistemáticamente con el nivel de precio y $|r|$ no, esto favorece el uso de retornos para una mejor generalización temporal.

**Objetivo:**  
Verificar si el target presenta cambios significativos según el horario intradía o el nivel de precio del instrumento.


### 3.2.1 Variación por hora / minuto del día

> ¿cambia la escala o el sesgo del target según el horario?

**Idea**

Queremos responder preguntas simples:
- ¿El target es más volátil en la apertura que en el cierre?
- ¿Hay sesgo direccional por horario?
- ¿Delta y retorno reaccionan distinto al régimen horario?

**Preparación**

Usaremos:
- `minute_of_day` (si ya lo tiene) o
- lo derivamos del índice datetime.

In [23]:
import pandas as pd
import numpy as np

# Asegurar columna minute_of_day (minutos desde apertura NY)
if "minute_of_day" not in mnq_intraday_targets.columns:
    dt = mnq_intraday_targets.index
    mnq_intraday_targets["minute_of_day"] = dt.hour * 60 + dt.minute

#### Estadísticos por minuto del día

Calculamos mediana y dispersión robusta por minuto:

In [24]:
def intraday_stats_by_minute(
    df: pd.DataFrame,
    *,
    target: str,
    minute_col: str = "minute_of_day",
):
    """
    Estadísticos intradía por minuto:
      - mediana
      - IQR
      - std
    """
    g = df.dropna(subset=[target]).groupby(minute_col)[target]

    out = g.agg(
        median="median",
        std="std",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
    )

    out["iqr"] = out["q75"] - out["q25"]
    return out


Ejecución:

In [25]:
stats_delta_60_min = intraday_stats_by_minute(
    mnq_intraday_targets,
    target="delta_60",
)

stats_delta_90_min = intraday_stats_by_minute(
    mnq_intraday_targets,
    target="delta_90",
)


stats_ret_60_min = intraday_stats_by_minute(
    mnq_intraday_targets,
    target="ret_60",
)

stats_ret_90_min = intraday_stats_by_minute(
    mnq_intraday_targets,
    target="ret_90",
)

stats_lret_60_min = intraday_stats_by_minute(
    mnq_intraday_targets,
    target="lret_60",
)

stats_lret_90_min = intraday_stats_by_minute(
    mnq_intraday_targets,
    target="lret_90",
)



In [26]:
mnq_intraday_targets

,date,open,high,low,close,volume,delta_60,ret_60,lret_60,delta_90,ret_90,lret_90,minute_of_day
datetime,,,,,,,,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7,9.00,0.001031,0.001031,6.00,0.000687,0.000687,390
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89,9.25,0.001060,0.001059,7.50,0.000859,0.000859,391
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34,9.75,0.001117,0.001117,8.00,0.000917,0.000916,392
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53,8.50,0.000974,0.000974,8.00,0.000917,0.000916,393
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3,8.00,0.000917,0.000916,7.75,0.000888,0.000888,394
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 15:56:00-04:00,2025-06-13,21624.50,21635.00,21613.50,21617.50,3251,NaN,NaN,NaN,NaN,NaN,NaN,956
2025-06-13 15:57:00-04:00,2025-06-13,21616.50,21635.25,21615.75,21623.75,2201,NaN,NaN,NaN,NaN,NaN,NaN,957
2025-06-13 15:58:00-04:00,2025-06-13,21623.25,21632.75,21616.50,21621.75,1859,NaN,NaN,NaN,NaN,NaN,NaN,958


#### 3.2.1.a Segmentación intradía basada en minute_of_day


**Definición de segmentos (NY time)**

Usando minutos desde medianoche (minute_of_day = hour*60 + minute):

- closed: 06:30 – 08:00
  - 390 ≤ minute < 480

- pre_market: 08:00 – 09:00
  - 480 ≤ minute < 540

- open: 09:00 – 10:00
  - 540 ≤ minute < 600

- mid_day: 10:00 – 15:00
  - 600 ≤ minute < 900

- close: 15:00 – 16:00
  - 900 ≤ minute ≤ 960

In [27]:
import pandas as pd

def add_intraday_segment_by_minute(
    stats_df: pd.DataFrame,
    *,
    minute_col_name: str = "minute_of_day",
) -> pd.DataFrame:
    """
    Agrega una columna 'segment' a una tabla de estadísticas intradía
    indexada por minute_of_day.

    Segmentos:
      - closed      : 06:30–08:00
      - pre_market  : 08:00–09:00
      - open        : 09:00–10:00
      - mid_day     : 10:00–15:00
      - close       : 15:00–16:00
    """
    out = stats_df.copy()

    # El índice ya representa minute_of_day
    out[minute_col_name] = out.index.astype(int)

    out["segment"] = "mid_day"

    out.loc[(out[minute_col_name] >= 390) & (out[minute_col_name] < 480), "segment"] = "closed"
    out.loc[(out[minute_col_name] >= 480) & (out[minute_col_name] < 540), "segment"] = "pre_market"
    out.loc[(out[minute_col_name] >= 540) & (out[minute_col_name] < 600), "segment"] = "open"
    out.loc[(out[minute_col_name] >= 600) & (out[minute_col_name] < 900), "segment"] = "mid_day"
    out.loc[(out[minute_col_name] >= 900) & (out[minute_col_name] <= 960), "segment"] = "close"

    return out


**Resumir por segmento (tablas pequeñas y legibles)***

- Reutilizamos el resumen por segmento:

In [28]:
def summarize_by_segment(
    stats_df_with_seg: pd.DataFrame,
    *,
    cols=("median", "std", "iqr"),
):
    """
    Resume estadísticas intradía por segmento horario.
    """
    return (
        stats_df_with_seg
        .groupby("segment")[list(cols)]
        .mean()
        .reindex(["closed", "pre_market", "open", "mid_day", "close"])
    )


**Ejecución completa** (ejemplo H = 60)

In [29]:
#60min
# Delta
stats_delta_60_seg = add_intraday_segment_by_minute(stats_delta_60_min)
summary_delta_60_seg = summarize_by_segment(stats_delta_60_seg)
# Retorno
stats_ret_60_seg = add_intraday_segment_by_minute(stats_ret_60_min)
summary_ret_60_seg = summarize_by_segment(stats_ret_60_seg)
# Log-retorno
stats_lret_60_seg = add_intraday_segment_by_minute(stats_lret_60_min)
summary_lret_60_seg = summarize_by_segment(stats_lret_60_seg)

#90min

stats_delta_90_seg = add_intraday_segment_by_minute(stats_delta_90_min)
summary_delta_90_seg = summarize_by_segment(stats_delta_90_seg)

stats_ret_90_seg = add_intraday_segment_by_minute(stats_ret_90_min)
summary_ret_90_seg = summarize_by_segment(stats_ret_90_seg)

stats_lret_90_seg = add_intraday_segment_by_minute(stats_lret_90_min)
summary_lret_90_seg = summarize_by_segment(stats_lret_90_seg)

In [30]:
print('\nSummary_delta_60_seg:\n')
display(summary_delta_60_seg)
print('\nSummary_ret_60_seg:\n')
display(summary_ret_60_seg)
print('\nSummary_lret_60_seg:\n')
display(summary_lret_60_seg)

print('\nSummary_delta_90_seg:\n')
display(summary_delta_90_seg)
print('\nSummary_ret_90_seg:\n')
display(summary_ret_90_seg)
print('\nSummary_lret_90_seg:\n')
display(summary_lret_90_seg)


Summary_delta_60_seg:



,median,std,iqr
segment,,,
closed,0.897222,38.761548,33.547222
pre_market,0.766667,57.956659,55.400000
open,4.629167,83.370967,96.579167
mid_day,3.726667,59.042902,56.212917
close,-0.750000,66.259716,57.625000



Summary_ret_60_seg:



,median,std,iqr
segment,,,
closed,0.000062,0.002936,0.002308
pre_market,0.000052,0.004287,0.003861
open,0.000322,0.005866,0.006761
mid_day,0.000261,0.004182,0.003902
close,-0.000051,0.004900,0.004052



Summary_lret_60_seg:



...

,median,std,iqr
segment,,,
closed,0.369444,51.652774,43.758333
pre_market,2.212500,81.391886,89.237500
open,6.366667,97.354609,113.975000
mid_day,5.020295,72.392816,67.809963
close,NaN,NaN,NaN



Summary_ret_90_seg:



,median,std,iqr
segment,,,
closed,0.000025,0.003885,0.003031
pre_market,0.000149,0.005868,0.006264
open,0.000444,0.006888,0.007942
mid_day,0.000349,0.005082,0.004723
close,NaN,NaN,NaN



Summary_lret_90_seg:



,median,std,iqr
segment,,,
closed,0.000025,0.003881,0.003031
pre_market,0.000149,0.005866,0.006263
open,0.000444,0.006890,0.007940
mid_day,0.000348,0.005071,0.004722
close,NaN,NaN,NaN


#### 3.2.1.b Observaciones -  Estabilidad intradía por segmento (3.2.1)


**Observaciones generales**

- En todos los casos, la **apertura de mercado (09:00–10:00)** es el segmento con **mayor dispersión** y **mayor magnitud típica** del movimiento futuro.
- El comportamiento intradía no es homogéneo: existen **regímenes horarios claramente diferenciados**.
- Retornos y log-retornos muestran resultados prácticamente idénticos, por lo que pueden considerarse equivalentes en este análisis.

---

**Horizonte H = 60 minutos**

**Delta (delta_60):**
- La dispersión (`std` e `iqr`) aumenta de forma marcada desde `pre_market` y alcanza su máximo en `open`.
- La mediana en `open` es significativamente mayor que en el resto del día, indicando movimientos más amplios pero también más ruidosos.
- El segmento `closed` presenta la menor dispersión, confirmando que el mercado está relativamente inactivo antes de las 08:00.
- En `close` aparece un leve sesgo negativo y alta dispersión, reflejando mayor inestabilidad al final de la sesión.

**Retorno / Log-retorno (ret_60 / lret_60):**
- La dispersión aumenta en apertura, pero **de forma más moderada** que en delta.
- La estructura intradía es más suave: las diferencias entre `pre_market`, `open` y `mid_day` son menos extremas.
- El sesgo direccional sigue el mismo patrón que en delta, pero con menor amplificación.

---

**Horizonte H = 90 minutos**

**Delta (delta_90):**
- El patrón se intensifica respecto a H=60: la apertura muestra la **mayor mediana y la mayor dispersión**.
- La volatilidad en `pre_market` y `open` es considerablemente mayor que en `mid_day`.
- No hay datos suficientes para el segmento `close`, lo cual es esperable por el horizonte de predicción.

**Retorno / Log-retorno (ret_90 / lret_90):**
- La apertura sigue siendo el segmento más activo, pero la dispersión crece de manera más controlada que en delta.
- La transición entre segmentos es más gradual, indicando **mayor estabilidad estadística**.
- El sesgo positivo en apertura es consistente, pero menos extremo que en puntos.

---

**Comparación delta vs retorno**

- El **delta en puntos** amplifica fuertemente los regímenes horarios: apertura y pre-market muestran picos claros de dispersión.
- Los **retornos** preservan la estructura intradía, pero con una respuesta más estable y menos dependiente del horario.
- A mayor horizonte (90 vs 60), esta diferencia se acentúa: el delta se vuelve más sensible al régimen horario, mientras que el retorno mantiene mayor regularidad.

#### **Conclusión del punto 3.2.1**




- El comportamiento del target **varía significativamente a lo largo del día**, especialmente en la apertura de mercado.  
- Los **deltas en puntos** capturan mejor la magnitud económica de los movimientos, pero son más sensibles al régimen horario y al aumento de volatilidad en apertura.  
- Los **retornos y log-retornos** muestran una estructura intradía más estable, lo que sugiere una mejor capacidad de generalización cuando el modelo debe aprender patrones a lo largo de distintos segmentos del día.

Este resultado justifica avanzar hacia el análisis de **variación entre jornadas (3.2.2)** y, posteriormente, evaluar si la apertura requiere un tratamiento específico en el diseño del target o del modelo.

#### Nota metodológica importante



A partir del análisis intradía realizado, **no es posible aún definir de forma concluyente** si el target de predicción debe formularse en términos de **deltas en puntos** o **retornos**.

Los resultados del punto 3.2.1 muestran que:
- los **deltas** reaccionan con mayor intensidad a los distintos regímenes horarios, especialmente en la apertura,
- los **retornos** presentan una estructura intradía más suave y estable.

Esto **no implica** que los deltas sean un objetivo inferior ni que los retornos deban adoptarse automáticamente como target final.  
Sí indica, sin embargo, una **inclinación preliminar** hacia los retornos desde el punto de vista de **estabilidad estadística**, que deberá ser confirmada (o refutada) en los próximos análisis.

La decisión final sobre el target se tomará únicamente después de evaluar:
- la variación entre jornadas (punto 3.2.2),
- la relación con el nivel de precio,
- y la predecibilidad empírica mediante baselines comparables (sección 3.3).

Hasta entonces, ambas formulaciones continúan siendo consideradas válidas.

### 3.2.2 Variación entre jornadas

>¿hay días “tranquilos” y días “explosivos”?

**Idea**

Analizar el target agregado por día, no por observación:
- volatilidad diaria del target,
- sesgo diario,
- estabilidad entre sesiones.

#### Estadísticos diarios

In [31]:
def daily_target_stats(
    df: pd.DataFrame,
    *,
    target: str,
    date_col: str = "date",
):
    """
    Estadísticos diarios del target:
      - mediana diaria
      - std diaria
      - IQR diaria
    """
    g = df.dropna(subset=[target]).groupby(date_col)[target]

    out = g.agg(
        median="median",
        std="std",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
    )

    out["iqr"] = out["q75"] - out["q25"]
    return out


In [32]:
daily_delta_60 = daily_target_stats(
    mnq_intraday_targets,
    target="delta_60",
)

daily_ret_60 = daily_target_stats(
    mnq_intraday_targets,
    target="ret_60",
)

daily_lret_60 = daily_target_stats(
    mnq_intraday_targets,
    target="lret_60",
)

daily_delta_90 = daily_target_stats(
    mnq_intraday_targets,
    target="delta_90",
)

daily_ret_90 = daily_target_stats(
    mnq_intraday_targets,
    target="ret_90",
)

daily_lret_90 = daily_target_stats(
    mnq_intraday_targets,
    target="lret_90",
)

#### **3.2.2.a Qué queremos responder**



Para cada target (delta / ret / lret) y horizonte (60 / 90):

¿Qué tan variable es la volatilidad de un día a otro?

¿El nivel típico diario (mediana) cambia mucho entre días?

¿Delta y retorno reaccionan distinto frente a días “tranquilos” vs “explosivos”?

#### **3.2.2.b Reducir “muchos días” a pocas métricas**



Cada daily_* tiene una fila por día.
Lo primero es resumir esa distribución de días.


Función de resumen interdiario:

In [33]:
def summarize_daily_distribution(
    daily_df: pd.DataFrame,
):
    """
    Resume la distribución de estadísticas diarias.
    Espera columnas: median, std, iqr.
    """
    return pd.DataFrame({
        "median_of_median": [daily_df["median"].median()],
        "std_of_median":    [daily_df["median"].std()],
        "median_of_std":    [daily_df["std"].median()],
        "std_of_std":       [daily_df["std"].std()],
        "median_of_iqr":    [daily_df["iqr"].median()],
        "std_of_iqr":       [daily_df["iqr"].std()],
    })


#### 3.2.2.c Construir tabla comparativa (delta vs retorno)

In [34]:
summary_daily = pd.concat(
    {
        "delta_60": summarize_daily_distribution(daily_delta_60),
        "ret_60":   summarize_daily_distribution(daily_ret_60),
        "lret_60":  summarize_daily_distribution(daily_lret_60),
        "delta_90": summarize_daily_distribution(daily_delta_90),
        "ret_90":   summarize_daily_distribution(daily_ret_90),
        "lret_90":  summarize_daily_distribution(daily_lret_90),
    }
).reset_index(level=0).rename(columns={"level_0": "target"})

display(summary_daily)

,target,median_of_median,std_of_median,median_of_std,std_of_std,median_of_iqr,std_of_iqr
0,delta_60,2.750000,17.883559,42.899685,27.520760,52.625000,33.793247
0,ret_60,0.000171,0.001277,0.002966,0.002076,0.003651,0.002603
0,lret_60,0.000171,0.001277,0.002970,0.002070,0.003649,0.002602
0,delta_90,3.750000,28.937957,49.913100,33.879250,66.750000,44.097451
0,ret_90,0.000251,0.002046,0.003490,0.002510,0.004556,0.003326
0,lret_90,0.000251,0.002046,0.003485,0.002501,0.004552,0.003324


#### 3.2.2.d Qué Observar (criterios concretos)

1. Estabilidad del nivel diario

- Compare: `std_of_median`
  - mide cuánto cambia el “centro” del target de un día a otro.

  Si delta tiene std_of_median mucho mayor que retorno →
  el nivel diario depende más del día (menos estable).

2. Estabilidad de la volatilidad diaria

- Compare: `std_of_std`
    - mide cuánto cambia la volatilidad diaria entre días.

  Si delta ≫ retorno →
  días muy tranquilos vs días muy explosivos → más difícil de aprender.

3. Robustez central

- Compare: `median_of_std` y `median_of_iqr`
  - escala típica diaria del target.

  Si delta crece fuerte con el horizonte y retorno no → señal de dependencia estructural.

#### 3.2.2.e. Observaciones -  Variación entre jornadas (3.2.2)


- Existen **diferencias claras entre días**, tanto en el nivel típico del movimiento como en su volatilidad.
- Retornos y log-retornos presentan resultados prácticamente idénticos, por lo que pueden considerarse equivalentes en este análisis.

---

**Horizonte H = 60 minutos**

**Delta (delta_60):**
- La **mediana diaria** del movimiento (median_of_median) es moderada, pero presenta una **alta variabilidad entre días** (`std_of_median` ≈ 17.9 pts).
- La **volatilidad diaria** (`median_of_std`) es elevada y cambia considerablemente de un día a otro (`std_of_std` ≈ 27.5 pts).
- El rango intercuartílico diario también muestra **alta dispersión interdiaria**, indicando días muy distintos entre sí.

**Retorno / Log-retorno (ret_60 / lret_60):**
- El nivel típico diario es cercano a cero y **mucho más estable entre jornadas**.
- Tanto la volatilidad diaria como su variación entre días son **significativamente menores** que en delta.
- La estructura diaria es más homogénea, con menor dependencia de días extremos.

---

**Horizonte H = 90 minutos**

**Delta (delta_90):**
- Aumenta la **variabilidad entre días** respecto a H=60, tanto en el nivel típico como en la volatilidad.
- La dispersión interdiaria se intensifica, lo que sugiere mayor sensibilidad a eventos diarios específicos.

**Retorno / Log-retorno (ret_90 / lret_90):**
- Aunque la dispersión diaria crece respecto a H=60, el crecimiento es **más controlado** que en delta.
- La variabilidad entre jornadas sigue siendo más baja en comparación con los deltas.
- La relación entre días permanece más estable.

---

**Comparación delta vs retorno**

- Los **deltas en puntos** muestran una **fuerte dependencia del día específico**, con diferencias marcadas entre jornadas tranquilas y jornadas explosivas.
- Los **retornos** presentan una **mayor estabilidad interdiaria**, tanto en nivel como en volatilidad.
- Esta diferencia se acentúa al aumentar el horizonte de predicción.


#### 3.2.2.f. Conclusión del punto 3.2.2

El análisis interdiario indica que los **retornos y log-retornos son más estables entre jornadas** que los deltas en puntos, reduciendo la dependencia del target respecto a días extremos o regímenes particulares.  

Este resultado **no descarta el uso de deltas**, pero refuerza la inclinación preliminar hacia los **retornos** desde el punto de vista de **robustez temporal**, a la espera de completar el análisis con la relación entre el target y el nivel de precio.

### 3.2.3 Relación con el nivel de precio

**Idea central**

Un aspecto clave para definir el target de predicción es evaluar su **dependencia respecto al nivel de precio** del instrumento.

Si el **delta en puntos** depende del nivel del precio, entonces:
- un mismo patrón aprendido en un período de precios bajos (por ejemplo 2019),
- puede no generalizar correctamente a períodos de precios más altos (por ejemplo 2024–2025).

En contraste, los **retornos**, al estar normalizados por el precio, deberían ser **más invariantes al nivel de precio** y, por lo tanto, más robustos frente a cambios estructurales de largo plazo.

---

**Metodología**

Se analiza la relación entre el **nivel de precio** (`close`) y la **magnitud absoluta del target** (riesgo, no dirección):

- correlación entre $|\Delta P_{t,h}|$ y `close`
- correlación entre $|r_{t,h}|$ y `close`

El uso del valor absoluto permite evaluar si la **escala del movimiento futuro** crece o decrece sistemáticamente con el nivel de precio.


#### Cálculo de correlaciones

Trabajamos con magnitud absoluta (riesgo, no dirección):

In [35]:
def correlation_with_price_level(
    df: pd.DataFrame,
    *,
    target: str,
    price_col: str = "close",
):
    """
    Correlación entre |target| y nivel de precio.
    """
    sub = df[[target, price_col]].dropna()
    return sub[target].abs().corr(sub[price_col])


Ejecución:

In [36]:
corr_delta_60 = correlation_with_price_level(
    mnq_intraday_targets,
    target="delta_60",
)

corr_ret_60 = correlation_with_price_level(
    mnq_intraday_targets,
    target="ret_60",
)

corr_delta_90 = correlation_with_price_level(
    mnq_intraday_targets,
    target="delta_90",
)

corr_ret_90 = correlation_with_price_level(
    mnq_intraday_targets,
    target="ret_90",
)

In [37]:
print("H = 60 minutos")
print(f'corr_delta_60:\t{corr_delta_60} ')
print(f'corr_ret_60:\t{corr_ret_60}')

print("\nH = 90 minutos")

print(f'corr_delta_90:\t{corr_delta_90}')
print(f'corr_ret_90:\t{corr_ret_90}')


H = 60 minutos
corr_delta_60:	0.0754337020652504 
corr_ret_60:	-0.15160332527828596

H = 90 minutos
corr_delta_90:	0.08229462598576713
corr_ret_90:	-0.15097892175945585


**Interpretación**

- El **delta en puntos** muestra una **correlación positiva**, aunque moderada, con el nivel de precio.  
  Esto indica que, a medida que el MNQ cotiza a niveles más altos, la magnitud típica de los movimientos en puntos tiende a aumentar.

- Los **retornos**, en cambio, presentan una **correlación negativa** con el nivel de precio.  
  Esto sugiere que la escala relativa de los movimientos no crece con el precio y tiende a mantenerse más estable en términos porcentuales.

- Este comportamiento es consistente en ambos horizontes (60 y 90 minutos), lo que refuerza la robustez del resultado.


**Implicancia para la definición del target**

Estos resultados indican que:

- el **delta en puntos** está parcialmente condicionado por el nivel de precio del activo,
- mientras que los **retornos son más invariantes al nivel de precio**.

Desde una perspectiva de **generalización temporal**, esto favorece el uso de retornos como target de predicción, ya que reduce el riesgo de que el modelo aprenda patrones dependientes de un régimen de precios específico.

#### Conclusión del punto 3.2.3


El análisis de correlación con el nivel de precio refuerza la inclinación preliminar hacia los **retornos** como objetivo de predicción, debido a su **mayor invariancia frente a cambios estructurales del precio**.  

Este resultado no invalida el uso de deltas en puntos, pero sí sugiere que, para modelos entrenados en largos períodos históricos, los retornos ofrecen una base más robusta para la generalización.

### **Cierre del punto 3.2 - Estabilidad del target en el tiempo**

El análisis de la estabilidad del target se abordó desde tres perspectivas complementarias: **variación intradía (3.2.1)**, **variación entre jornadas (3.2.2)** y **relación con el nivel de precio (3.2.3)**.

En conjunto, los resultados muestran que el comportamiento del target **no es homogéneo en el tiempo**:

- **Intradiariamente**, existen regímenes horarios bien definidos, con una amplificación clara de la dispersión en la apertura de mercado. Este efecto es más pronunciado en los **deltas en puntos**, mientras que los **retornos** presentan una estructura más estable a lo largo del día.
- **Entre jornadas**, los deltas exhiben una mayor variabilidad tanto en el nivel típico diario como en la volatilidad, reflejando una fuerte dependencia de días extremos. Los retornos, en cambio, muestran una **mayor homogeneidad interdiaria**.
- **Respecto al nivel de precio**, los deltas presentan una correlación positiva con el precio del activo, mientras que los retornos son significativamente más invariantes, lo que favorece su capacidad de generalización a lo largo de distintos regímenes históricos.

En síntesis, el punto 3.2 no define aún de manera concluyente el target final, pero aporta evidencia consistente de que los **retornos ofrecen mayor estabilidad temporal y estructural**, mientras que los **deltas en puntos capturan mejor la magnitud económica, a costa de una mayor sensibilidad a régimen horario, día específico y nivel de precio**.

Esta sección deja planteado el trade-off central entre **interpretabilidad operativa** y **robustez estadística**, y habilita avanzar al siguiente paso: evaluar la **predecibilidad empírica** de ambos targets mediante baselines comparables (sección 3.3).



## 3.3. Predecibilidad preliminar (sin modelos complejos)

#### Objetivo

Antes de entrenar redes neuronales u otros modelos complejos, el objetivo es responder una pregunta muy concreta:

- ¿Existe señal temporal explotable en el target y cuál formulación (delta o retorno) la preserva mejor?

Para eso se evalúan tres cosas, en orden creciente de complejidad:

1. Dependencia temporal intrínseca del target (ACF).
2. Capacidad predictiva de baselines muy simples.
3. Comparación justa delta vs retorno en puntos.

### 3.3.1 Autocorrelación del target (diagnóstico puro)

Qué buscamos

- Si el target es ruido blanco, ningún modelo aprenderá algo estable.
- Si existe autocorrelación débil pero persistente, hay señal potencial.

Se analiza:
- el target crudo
- el valor absoluto del target (persistencia de volatilidad)-

**Código: ACF del target y |target|**

In [38]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import acf

def compute_acf_summary(
    series: pd.Series,
    *,
    nlags: int = 30,
):
    """
    Calcula ACF hasta nlags y devuelve un DataFrame.
    """
    x = series.dropna().values
    acf_vals = acf(x, nlags=nlags, fft=True)
    return pd.DataFrame({
        "lag": np.arange(len(acf_vals)),
        "acf": acf_vals,
    })


In [39]:
acf_delta_60 = compute_acf_summary(mnq_intraday_targets["delta_60"])
acf_ret_60   = compute_acf_summary(mnq_intraday_targets["ret_60"])

acf_abs_delta_60 = compute_acf_summary(mnq_intraday_targets["delta_60"].abs())
acf_abs_ret_60   = compute_acf_summary(mnq_intraday_targets["ret_60"].abs())

acf_delta_90 = compute_acf_summary(mnq_intraday_targets["delta_90"])
acf_ret_90   = compute_acf_summary(mnq_intraday_targets["ret_90"])

acf_abs_delta_90 = compute_acf_summary(mnq_intraday_targets["delta_90"].abs())
acf_abs_ret_90   = compute_acf_summary(mnq_intraday_targets["ret_90"].abs())

In [40]:
import pandas as pd

# 1) Diccionario: nombre_columna -> df_acf (cada df con columnas: lag, acf)
acf_map = {
    "acf_delta_60": acf_delta_60,
    "acf_ret_60": acf_ret_60,
    "acf_abs_delta_60": acf_abs_delta_60,
    "acf_abs_ret_60": acf_abs_ret_60,
    "acf_delta_90": acf_delta_90,
    "acf_ret_90": acf_ret_90,
    "acf_abs_delta_90": acf_abs_delta_90,
    "acf_abs_ret_90": acf_abs_ret_90,
}

# =========================
# A) Formato ancho (wide)
# =========================
dfs_wide = []
for name, df in acf_map.items():
    tmp = df.rename(columns={"acf": name})
    dfs_wide.append(tmp)

acf_wide = dfs_wide[0]
for tmp in dfs_wide[1:]:
    acf_wide = acf_wide.merge(tmp, on="lag", how="inner")  # o "outer" si hubiera lags distintos

acf_wide = acf_wide.sort_values("lag").reset_index(drop=True)

# =========================
# B) Formato largo (long/tidy)
# =========================
acf_long = pd.concat(
    [df.assign(series=name) for name, df in acf_map.items()],
    ignore_index=True
).rename(columns={"acf": "acf_value"})

acf_long = acf_long[["lag", "series", "acf_value"]].sort_values(["lag", "series"]).reset_index(drop=True)


In [41]:
acf_wide

,lag,acf_delta_60,acf_ret_60,acf_abs_delta_60,acf_abs_ret_60,acf_delta_90,acf_ret_90,acf_abs_delta_90,acf_abs_ret_90
0,0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,1,0.982118,0.981703,0.971354,0.971330,0.987679,0.987211,0.979830,0.979443
2,2,0.964039,0.963184,0.944222,0.944101,0.975254,0.974262,0.960557,0.959697
3,3,0.945986,0.944826,0.918322,0.918327,0.962895,0.961485,0.942100,0.941051
4,4,0.928138,0.926701,0.893982,0.894078,0.950637,0.948768,0.924362,0.923061
5,5,0.910238,0.908549,0.870445,0.870697,0.938285,0.936055,0.906850,0.905507
6,6,0.892605,0.890608,0.847985,0.848317,0.926076,0.923410,0.889865,0.888435
7,7,0.875108,0.872797,0.826280,0.826697,0.913939,0.910872,0.873304,0.871775
8,8,0.857744,0.855033,0.805324,0.805637,0.901927,0.898443,0.857276,0.855639
9,9,0.840504,0.837341,0.784996,0.785200,0.889917,0.885980,0.841517,0.839721


Qué debe mirar (sin interpretar aún)

- ACF del target crudo:
  - ¿cae rápidamente a ~0?

- ACF de |target|:
  - ¿decay lento? → persistencia de volatilidad (típico en mercados)

Esto no decide el target, solo confirma si hay estructura temporal mínima.

#### Conclusiones del análisis de autocorrelación (ACF)



1. **Target crudo (delta y retorno)**
   - No se observa una caída rápida de la ACF hacia valores cercanos a cero; la autocorrelación se mantiene elevada incluso hasta lag 30.
   - El decaimiento de la ACF es suave y aproximadamente lineal, sin rupturas abruptas.
   - Las curvas de `delta_*` y `ret_*` son prácticamente idénticas en todos los lags analizados.
   - El target crudo no se comporta como ruido blanco y presenta dependencia temporal persistente.

2. **Valor absoluto del target (|target|)**
   - La ACF del valor absoluto muestra un decaimiento aún más lento que el del target crudo.
   - Se observa una estructura de autocorrelación estable y persistente a lo largo de todos los lags.
   - No se identifican diferencias relevantes entre delta y retorno en términos de persistencia del |target|.

3. **Implicancia del análisis**
   - Existe evidencia de estructura temporal mínima tanto en el target crudo como en su valor absoluto.
   - Este análisis es exploratorio y no define aún la elección del target, pero confirma que ambos contienen dependencia temporal potencialmente explotable.


### 3.3.2 Baselines simples y comparables

Aquí empezamos a medir capacidad predictiva real, pero con modelos que:

- no sobreajustan,
- son rápidos,
- y sirven como referencia dura.

**Baselines a usar**

1. Naive
- Zero: predice 0 siempre
- Last value: predice el último valor observado
- Mean: predice la media histórica del target

2. Modelo lineal simple
- Ridge Regression
- (opcional luego: MLP muy pequeño)

**Importante: mismo split temporal**

  Se usa exactamente el mismo split temporal para:
  - delta
  - retorno

Nada de cross-validation aleatoria.

#### Paso 1 (ajustado): crear un split temporal simple

Objetivo de este paso
- Tener TRAIN / VALID / TEST coherentes en el tiempo.
- Usarlos igual para delta y retorno.
- Solo para baselines Naive.

##### Paso 1.1 — Definir un split temporal simple


Usamos proporciones típicas y orden temporal estricto:
- 70% → train
- 15% → valid
- 15% → test

In [42]:
import numpy as np
import pandas as pd

def temporal_split_indices(
    df: pd.DataFrame,
    *,
    train_frac: float = 0.70,
    valid_frac: float = 0.15,
):
    """
    Devuelve índices (boolean masks) para train / valid / test
    respetando el orden temporal del DataFrame.
    """
    n = len(df)
    i_train_end = int(n * train_frac)
    i_valid_end = int(n * (train_frac + valid_frac))

    idx = np.arange(n)

    train_mask = idx < i_train_end
    valid_mask = (idx >= i_train_end) & (idx < i_valid_end)
    test_mask  = idx >= i_valid_end

    return train_mask, valid_mask, test_mask


In [43]:
train_m, valid_m, test_m = temporal_split_indices(mnq_intraday_targets)


##### Paso 1.2 — Baseline Naive “zero” con split temporal

Reutilizamos el evaluador, pero pasando masks en lugar de columna split.

**Métricas**

In [46]:
def compute_basic_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    err = y_pred - y_true
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err**2)))

    sse = float(np.sum((y_true - y_pred)**2))
    sst = float(np.sum((y_true - np.mean(y_true))**2))
    r2 = float(1.0 - sse / sst) if sst > 0 else np.nan

    return {"MAE": mae, "RMSE": rmse, "R2": r2}


**Evaluador Naive Zero**

In [47]:
def eval_naive_zero_masks(
    df: pd.DataFrame,
    *,
    target_col: str,
    train_mask,
    valid_mask,
    test_mask,
):
    """
    Evalúa baseline naive_zero (predice 0) en valid y test.
    """
    rows = []

    for split_name, mask in [
        ("valid", valid_mask),
        ("test",  test_mask),
    ]:
        sub = df.loc[mask, [target_col]].dropna()
        y_true = sub[target_col].to_numpy()
        y_pred = np.zeros_like(y_true)

        metrics = compute_basic_metrics(y_true, y_pred)

        rows.append({
            "model": "naive_zero",
            "split": split_name,
            "target": target_col,
            **metrics
        })

    return pd.DataFrame(rows)


##### Paso 1.3 — Ejecutar baseline zero (H=60 y H=90)

In [48]:
res_zero = pd.concat([
    eval_naive_zero_masks(
        mnq_intraday_targets,
        target_col="delta_60",
        train_mask=train_m,
        valid_mask=valid_m,
        test_mask=test_m,
    ),
    eval_naive_zero_masks(
        mnq_intraday_targets,
        target_col="ret_60",
        train_mask=train_m,
        valid_mask=valid_m,
        test_mask=test_m,
    ),
    eval_naive_zero_masks(
        mnq_intraday_targets,
        target_col="delta_90",
        train_mask=train_m,
        valid_mask=valid_m,
        test_mask=test_m,
    ),
    eval_naive_zero_masks(
        mnq_intraday_targets,
        target_col="ret_90",
        train_mask=train_m,
        valid_mask=valid_m,
        test_mask=test_m,
    ),
], ignore_index=True)

display(res_zero)


,model,split,target,MAE,RMSE,R2
0,naive_zero,valid,delta_60,36.201270,51.227930,-7.577321e-04
1,naive_zero,test,delta_60,54.699733,84.947602,-1.217315e-05
2,naive_zero,valid,ret_60,0.002019,0.002841,-1.117700e-03
3,naive_zero,test,ret_60,0.002710,0.004363,-1.418935e-07
4,naive_zero,valid,delta_90,45.714158,64.439664,-1.165414e-03
5,naive_zero,test,delta_90,69.222776,106.370580,-9.748890e-07
6,naive_zero,valid,ret_90,0.002550,0.003570,-1.714539e-03
7,naive_zero,test,ret_90,0.003429,0.005476,-2.504130e-05


- Este baseline es el piso absoluto.
- Si luego:
  - naive_mean o ridge no mejoran esto → no hay señal.
-Compare delta vs retorno:
  - ¿Cuál tiene menor RMSE relativo?
  - ¿Cuál es más estable entre valid y test?

No sacar conclusiones todavía, solo verificar coherencia.

##### Observaciones - Baseline Naive “Zero” (Paso 3.3.2.1)

- **R² ≈ 0 en todos los casos**  
  Este resultado es el esperado: predecir siempre cero no explica variabilidad alguna.  
  El modelo sirve únicamente como **piso de referencia**.

- **Coherencia entre valid y test**  
  Las métricas mantienen el mismo orden de magnitud entre *validation* y *test*, lo que indica que el **split temporal es consistente** y no introduce sesgos artificiales.

- **Escala del error**
  - En **delta**, el RMSE aumenta de forma marcada al pasar de H=60 a H=90, reflejando la mayor dispersión natural del target en horizontes más largos.
  - En **retornos**, el crecimiento del error es más controlado y proporcional al horizonte.

- **Comparación delta vs retorno (preliminar)**  
  - Ambos targets parten de un baseline igualmente débil, como corresponde a un modelo naive.
  - No se observa aún una ventaja clara, aunque los **retornos muestran una escala de error más estable** al variar el horizonte.

---

**Qué concluye este paso (y qué no)**

- El baseline “zero” **funciona correctamente como referencia mínima**.  
- El esquema de división temporal es adecuado para continuar con la evaluación.  
- Este paso **no permite decidir** el target final.

Este baseline cumple un único rol metodológico:

> *Cualquier modelo que no mejore este resultado carece de valor predictivo.*


#### Paso 2: Baseline Naive mean

Qué hace este baseline
- Calcula la media del target en TRAIN.
- Predice ese valor constante para VALID y TEST.
- Usa el mismo split temporal que en el Paso 1

**1. Evaluador Naive “Mean” (con masks)**

In [49]:
import numpy as np
import pandas as pd

def eval_naive_mean_masks(
    df: pd.DataFrame,
    *,
    target_col: str,
    train_mask,
    valid_mask,
    test_mask,
):
    """
    Evalúa baseline naive_mean:
      y_pred = mean(target) calculada SOLO en TRAIN.
    """
    rows = []

    # Media del target en TRAIN
    train_vals = df.loc[train_mask, target_col].dropna().to_numpy()
    mean_train = float(train_vals.mean())

    for split_name, mask in [
        ("valid", valid_mask),
        ("test",  test_mask),
    ]:
        sub = df.loc[mask, [target_col]].dropna()
        y_true = sub[target_col].to_numpy()
        y_pred = np.full_like(y_true, mean_train, dtype=float)

        metrics = compute_basic_metrics(y_true, y_pred)

        rows.append({
            "model": "naive_mean",
            "split": split_name,
            "target": target_col,
            "mean_train": mean_train,
            **metrics
        })

    return pd.DataFrame(rows)


**2. Ejecutar para delta y retorno (H=60 y H=90)**

In [50]:
res_mean = pd.concat([
    eval_naive_mean_masks(
        mnq_intraday_targets,
        target_col="delta_60",
        train_mask=train_m,
        valid_mask=valid_m,
        test_mask=test_m,
    ),
    eval_naive_mean_masks(
        mnq_intraday_targets,
        target_col="ret_60",
        train_mask=train_m,
        valid_mask=valid_m,
        test_mask=test_m,
    ),
    eval_naive_mean_masks(
        mnq_intraday_targets,
        target_col="delta_90",
        train_mask=train_m,
        valid_mask=valid_m,
        test_mask=test_m,
    ),
    eval_naive_mean_masks(
        mnq_intraday_targets,
        target_col="ret_90",
        train_mask=train_m,
        valid_mask=valid_m,
        test_mask=test_m,
    ),
], ignore_index=True)

display(res_mean)


,model,split,target,mean_train,MAE,RMSE,R2
0,naive_mean,valid,delta_60,0.448673,36.165107,51.217547,-0.000352
1,naive_mean,test,delta_60,0.448673,54.680144,84.950353,-0.000077
2,naive_mean,valid,ret_60,0.000058,0.002015,0.002839,-0.000168
3,naive_mean,test,ret_60,0.000058,0.002708,0.004363,-0.000188
4,naive_mean,valid,delta_90,0.686808,45.658417,64.419889,-0.000551
5,naive_mean,test,delta_90,0.686808,69.189832,106.372119,-0.000030
6,naive_mean,valid,ret_90,0.000091,0.002543,0.003568,-0.000257
7,naive_mean,test,ret_90,0.000091,0.003426,0.005476,-0.000133


**3. Qué debe mirar (muy concreto)**


Compare naive_mean vs naive_zero:

1. ¿Mejora MAE / RMSE?

    - Si sí → existe sesgo promedio explotable.
    - Si no → el target es esencialmente centrado en cero.

2. R²

    - R² ligeramente positivo (o menos negativo que zero) ya es señal.
    - No espere valores grandes: esto sigue siendo un baseline.

3. Estabilidad valid vs test

    - Si mejora en valid pero empeora fuerte en test → no generaliza.

4. Delta vs retorno

    - ¿Cuál muestra mejora más consistente frente a zero?
    - Ese target preserva mejor la señal promedio.

**4. Observaciones – Baseline Naive “Mean” vs “Zero” (Paso 3.3.2.2)**

Se comparan los resultados del baseline **naive_mean** (predicción de la media del TRAIN) frente al **naive_zero**, utilizando el mismo split temporal.

---

**Comparación general**

- En **todos los casos**, el modelo **naive_mean** muestra una **mejora marginal pero consistente** respecto a **naive_zero** en MAE y RMSE.
- Esta mejora confirma la existencia de un **sesgo promedio distinto de cero** en los targets, aunque de magnitud muy pequeña.
- Los valores de **R² permanecen cercanos a cero y negativos**, lo cual es esperable para baselines constantes.

---

**Horizonte H = 60 minutos**

**Delta (delta_60):**
- La mejora frente a naive_zero es **muy leve** tanto en valid como en test.
- El error absoluto y cuadrático prácticamente no cambian.
- El sesgo promedio en puntos es pequeño en relación con la dispersión total del target.

**Retorno (ret_60):**
- La mejora frente a naive_zero es **ligeramente más consistente** que en delta.
- El MAE y RMSE se reducen de forma estable en valid y test.
- El sesgo promedio capturado es pequeño, pero más coherente en términos relativos.

---

**Horizonte H = 90 minutos**

**Delta (delta_90):**
- Se observa una mejora marginal respecto a naive_zero, similar al caso H=60.
- La magnitud del error sigue dominada por la alta variabilidad del target.
- El sesgo promedio no aporta una reducción significativa del error.

**Retorno (ret_90):**
- La mejora frente a naive_zero es **consistente en ambos splits**.
- Aunque el R² sigue siendo cercano a cero, el comportamiento es más estable que en delta.
- El crecimiento del error al pasar de H=60 a H=90 es más controlado que en puntos.

---

**Comparación delta vs retorno**

- En ambos horizontes, los **retornos capturan el sesgo promedio de forma más estable** que los deltas en puntos.
- En los deltas, la media es pequeña frente a la dispersión, lo que limita su capacidad explicativa.
- En los retornos, aun siendo pequeños, los valores medios se traducen en **mejoras relativas más consistentes**.

---

**Conclusión del Paso 2**

El baseline **naive_mean** confirma la presencia de una **señal promedio débil**, insuficiente por sí sola para una predicción útil, pero:

- más **coherente y estable en retornos** que en deltas,
- consistente entre valid y test,
- y alineada con los resultados de estabilidad temporal analizados en la sección 3.2.

Este resultado no es decisivo, pero **refuerza la inclinación preliminar hacia los retornos** como formulación del target antes de avanzar a modelos lineales simples (Paso 3).


#### Paso 3: Ridge Regression (baseline lineal)

**Objetivo del paso**

Ver si un modelo lineal regularizado logra:
- mejorar a los baselines Naive,
- y cuál target (delta vs retorno) generaliza mejor con la misma información.

Importante: aquí no buscamos performance, sino comparabilidad y estabilidad.

#### Paso 3.1 — Definir el set de features (mínimo)

Para no mezclar efectos, usamos un baseline autoregresivo simple:

- el valor actual del target como única feature.

Esto responde:

> ¿Existe dependencia temporal lineal explotable?

#### Paso 3.2 — Preparar X e y (sin ventanas complejas)

In [51]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

1. Crear split temporal como Series (train/valid/test)

In [55]:
import numpy as np
import pandas as pd

def make_temporal_split_series(
    df: pd.DataFrame,
    *,
    train_frac: float = 0.70,
    valid_frac: float = 0.15,
    name: str = "split",
) -> pd.Series:
    """
    Crea una serie 'split' indexada por df.index (datetime),
    con valores: train / valid / test, respetando el orden temporal.
    """
    n = len(df)
    i_train_end = int(n * train_frac)
    i_valid_end = int(n * (train_frac + valid_frac))

    split = np.empty(n, dtype=object)
    split[:i_train_end] = "train"
    split[i_train_end:i_valid_end] = "valid"
    split[i_valid_end:] = "test"

    return pd.Series(split, index=df.index, name=name)

2. Preparar dataset AR(1)

In [56]:
def prepare_ar_dataset(
    df: pd.DataFrame,
    *,
    target_col: str,
    lag: int = 1,
):
    """
    Dataset autoregresivo simple:
      X_t = target_{t-lag}
      y_t = target_t
    """
    y = df[target_col]
    X = y.shift(lag)

    data = pd.concat([X.rename("x_lag"), y.rename("y")], axis=1).dropna()
    return data[["x_lag"]], data["y"]

#### Paso 3.3 — Entrenar Ridge (con escalado correcto)

El escalado se ajusta solo con TRAIN.

3. Ridge AR(1) alineado por índice

In [57]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

def eval_ridge_ar(
    df: pd.DataFrame,
    *,
    target_col: str,
    split_s: pd.Series,     # <- split indexado por datetime
    alpha: float = 1.0,
):
    """
    Ridge autoregresivo (lag=1), evaluado en valid y test.
    El split se alinea por índice (datetime), evitando errores de indexación.
    """
    rows = []

    # Dataset AR
    X, y = prepare_ar_dataset(df, target_col=target_col, lag=1)

    # Alinear split al índice del dataset AR (por el shift/dropna)
    split_aligned = split_s.loc[X.index]

    # Split
    X_train, y_train = X.loc[split_aligned == "train"], y.loc[split_aligned == "train"]
    X_valid, y_valid = X.loc[split_aligned == "valid"], y.loc[split_aligned == "valid"]
    X_test,  y_test  = X.loc[split_aligned == "test"],  y.loc[split_aligned == "test"]

    # Escalado (fit solo en TRAIN)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_valid_s = scaler.transform(X_valid)
    X_test_s  = scaler.transform(X_test)

    # Modelo
    model = Ridge(alpha=alpha)
    model.fit(X_train_s, y_train)

    for split_name, X_s, y_true in [
        ("valid", X_valid_s, y_valid),
        ("test",  X_test_s,  y_test),
    ]:
        y_pred = model.predict(X_s)
        metrics = compute_basic_metrics(y_true.to_numpy(), y_pred)

        rows.append({
            "model": "ridge_ar1",
            "split": split_name,
            "target": target_col,
            "alpha": alpha,
            **metrics
        })

    return pd.DataFrame(rows)



#### Paso 3.4 — Ejecutar Ridge para delta y retorno

4. Ejecutar (delta/ret, 60/90)

In [58]:
# Split temporal por índice (una sola vez)
split_s = make_temporal_split_series(mnq_intraday_targets, train_frac=0.70, valid_frac=0.15)

res_ridge = pd.concat([
    eval_ridge_ar(mnq_intraday_targets, target_col="delta_60", split_s=split_s, alpha=1.0),
    eval_ridge_ar(mnq_intraday_targets, target_col="ret_60",   split_s=split_s, alpha=1.0),
    eval_ridge_ar(mnq_intraday_targets, target_col="delta_90", split_s=split_s, alpha=1.0),
    eval_ridge_ar(mnq_intraday_targets, target_col="ret_90",   split_s=split_s, alpha=1.0),
], ignore_index=True)

display(res_ridge)



,model,split,target,alpha,MAE,RMSE,R2
0,ridge_ar1,valid,delta_60,1.0,6.344638,9.216015,0.967662
1,ridge_ar1,test,delta_60,1.0,9.723379,15.238656,0.967868
2,ridge_ar1,valid,ret_60,1.0,0.000354,0.000512,0.967485
3,ridge_ar1,test,ret_60,1.0,0.000482,0.000782,0.967896
4,ridge_ar1,valid,delta_90,1.0,6.533690,9.344644,0.978983
5,ridge_ar1,test,delta_90,1.0,9.966572,15.441234,0.978959
6,ridge_ar1,valid,ret_90,1.0,0.000365,0.000520,0.978812
7,ridge_ar1,test,ret_90,1.0,0.000494,0.000794,0.979023


#### Paso 3.5 — Qué debe mirar (muy concreto)

Compare Ridge vs Naive Mean:

1. ¿Mejora MAE / RMSE?
    - Si no mejora, la dependencia lineal es débil o inexistente.

2. R²
    - R² ligeramente positivo ya es señal.

3. Valid vs Test
    - Si mejora en valid pero cae en test → no generaliza.

4. Delta vs Retorno
    - ¿En cuál la mejora es más consistente?

#### Observaciones – Ridge AR(1) vs Baselines Naive (Paso 3.3.2.3)

Se comparan los resultados del modelo **Ridge autoregresivo (lag = 1)** frente al baseline **naive_mean**, utilizando el mismo split temporal y el mismo esquema para deltas y retornos.

---

**Comparación general**

- El modelo **Ridge AR(1)** mejora de forma **muy significativa** a los baselines naive en todos los casos.
- La mejora es consistente en **valid y test**, lo que indica **fuerte capacidad de generalización**.
- Los valores de **R² (~0.97–0.98)** confirman la presencia de una **dependencia temporal muy fuerte** en el target.

---

**Horizonte H = 60 minutos**

**Delta (delta_60):**
- Reducción drástica del error respecto a naive_mean:
  - RMSE pasa de ~51 a ~9 en valid y de ~85 a ~15 en test.
- El R² cercano a 0.97 indica que gran parte de la varianza del target se explica por su valor pasado.
- El comportamiento es estable entre valid y test.

**Retorno (ret_60):**
- Se observa una mejora proporcionalmente equivalente a la de delta.
- Los errores absolutos son pequeños y coherentes con la escala del target.
- R² similar al de delta, sin degradación en test.

---

**Horizonte H = 90 minutos**

**Delta (delta_90):**
- La mejora frente a naive_mean es aún más marcada que en H=60.
- R² cercano a 0.98 en valid y test, indicando fuerte persistencia temporal.
- La estabilidad entre splits es muy alta.

**Retorno (ret_90):**
- Resultados prácticamente idénticos a delta en términos relativos.
- Error absoluto bajo y consistente.
- Excelente generalización temporal.

---

**Comparación delta vs retorno**

- Desde el punto de vista de **predecibilidad temporal**, **ambos targets muestran una estructura extremadamente similar**.
- La dependencia temporal capturada por el modelo lineal es **igual de fuerte** en deltas y retornos.
- No se observa una ventaja clara de uno sobre otro en términos de **R² o estabilidad entre splits**.

---

**Interpretación metodológica clave**

El alto R² no implica que el problema esté “resuelto” ni que el modelo sea útil para trading real.  
Este resultado indica que:

- existe **fuerte autocorrelación de corto plazo** en el target,
- dicha estructura es capturable incluso por un modelo lineal muy simple,
- y **no depende de la formulación del target (delta vs retorno)** en esta etapa.

---

**Conclusión del Paso 3**

El análisis con Ridge AR(1) demuestra que:

- ambos targets son **claramente predecibles en sentido estadístico**,
- los retornos **no pierden señal** respecto a los deltas,
- y la decisión entre delta y retorno **no debe basarse en predecibilidad**, sino en criterios de:
  - estabilidad temporal,
  - invariancia al nivel de precio,
  - y coherencia económica.

Este resultado habilita cerrar la sección **3.3 Evaluación preliminar de predecibilidad** y avanzar hacia la definición final del target.


### 3.3.3 Evaluación doble (clave metodológica)

El objetivo de esta etapa es garantizar una **comparación justa** entre targets formulados en **deltas en puntos** y **retornos**, separando dos planos de evaluación:

1. **Calidad de aprendizaje en la escala propia del target**  
2. **Impacto económico real medido en puntos**


#### 3.3.3.a Métricas en el espacio del target


Estas métricas evalúan qué tan bien el modelo aprende el target **tal como fue formulado**:

- **MAE**
- **RMSE**
- **R²**

Este análisis responde a la pregunta:

> *¿El modelo es capaz de aprender una estructura estadística en esta escala?*

Es el plano donde:
- delta se evalúa en puntos,
- retorno se evalúa en unidades relativas.

Estas métricas **no permiten comparar delta vs retorno entre sí**, solo sirven para:
- comparar modelos dentro del mismo target,

#### 3.3.3.b Métricas en puntos (comparación económica justa)


Para comparar enfoques basados en **retornos** con aquellos basados directamente en **deltas**, se transforman las predicciones de retornos a puntos.

Para cada instante $ t $ y horizonte $ h $:

$$
\widehat{\Delta P}_{t,h} = P_t \cdot \widehat{r}_{t,h}
$$

donde:
- $ P_t $ es el precio actual (`close`),
- $ \widehat{r}_{t,h} $ es la predicción del retorno.

Luego se evalúan:

- **MAE en puntos**
- **RMSE en puntos**

Este plano responde a la pregunta clave:

> *¿Cuál formulación produce errores económicos menores, medidos en puntos reales?*


#### 3.3.3.c Implementación práctica


**Conversión de predicciones de retorno a puntos**

In [63]:
def returns_to_points(y_pred_ret: np.ndarray, price_t: np.ndarray) -> np.ndarray:
    """
    Convierte predicciones de retorno a puntos:
      ΔP_hat = P_t * r_hat
    """
    return price_t * y_pred_ret

**Evaluación económica en puntos**

In [64]:
def compute_point_metrics(y_true_points: np.ndarray, y_pred_points: np.ndarray) -> dict:
    """
    Métricas económicas en puntos.
    """
    y_true_points = np.asarray(y_true_points, dtype=float)
    y_pred_points = np.asarray(y_pred_points, dtype=float)

    err = y_pred_points - y_true_points
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err**2)))
    return {"MAE_pts": mae, "RMSE_pts": rmse}


**Función: obtener predicciones Ridge AR(1) alineadas (y precios)**

Esta función entrena Ridge AR(1) para un target_col y devuelve, para valid/test:
- y_true_target
- y_pred_target
- price_t (close actual)
- y_true_delta_points (si corresponde)

In [65]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

def ridge_ar_predictions(
    df: pd.DataFrame,
    *,
    target_col: str,
    split_s: pd.Series,
    alpha: float = 1.0,
    price_col: str = "close",
    lag: int = 1,
) -> pd.DataFrame:
    """
    Entrena Ridge AR(lag) sobre target_col y devuelve predicciones para valid/test.
    Dataset AR: X_t = target_{t-lag}, y_t = target_t.
    Incluye close_t para conversión a puntos si target es retorno.
    """
    # Construir dataset AR y alinear split
    y = df[target_col]
    X = y.shift(lag).rename("x_lag")
    data = pd.concat([X, y.rename("y"), df[price_col].rename("price_t")], axis=1).dropna()

    split_aligned = split_s.loc[data.index]

    # Split
    train_idx = split_aligned == "train"
    valid_idx = split_aligned == "valid"
    test_idx  = split_aligned == "test"

    X_train = data.loc[train_idx, ["x_lag"]]
    y_train = data.loc[train_idx, "y"]

    X_valid = data.loc[valid_idx, ["x_lag"]]
    y_valid = data.loc[valid_idx, "y"]
    p_valid = data.loc[valid_idx, "price_t"]

    X_test  = data.loc[test_idx, ["x_lag"]]
    y_test  = data.loc[test_idx, "y"]
    p_test  = data.loc[test_idx, "price_t"]

    # Escalado solo en TRAIN
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_valid_s = scaler.transform(X_valid)
    X_test_s  = scaler.transform(X_test)

    # Entrenar
    model = Ridge(alpha=alpha)
    model.fit(X_train_s, y_train)

    # Predecir
    y_pred_valid = model.predict(X_valid_s)
    y_pred_test  = model.predict(X_test_s)

    # Empaquetar
    out_valid = pd.DataFrame({
        "split": "valid",
        "target": target_col,
        "y_true": y_valid.to_numpy(),
        "y_pred": y_pred_valid,
        "price_t": p_valid.to_numpy(),
    }, index=y_valid.index)

    out_test = pd.DataFrame({
        "split": "test",
        "target": target_col,
        "y_true": y_test.to_numpy(),
        "y_pred": y_pred_test,
        "price_t": p_test.to_numpy(),
    }, index=y_test.index)

    return pd.concat([out_valid, out_test], axis=0)


**3) Ejecutar: predicciones para delta y retorno (H=60 como ejemplo)**

Puede repetir el bloque para H=90 cambiando columnas.

In [68]:
# Split temporal indexado por datetime (igual que antes)
split_s = make_temporal_split_series(mnq_intraday_targets, train_frac=0.70, valid_frac=0.15)

**4) Métricas en puntos: comparar retorno vs delta (H=60)**

Para comparar en puntos:
- Verdad en puntos: delta_60 real
- Predicción en puntos (modelo delta): y_pred de pred_delta_60
- Predicción en puntos (modelo retorno): price_t * y_pred de pred_ret_60

In [67]:
def compare_in_points(
    pred_delta: pd.DataFrame,
    pred_ret: pd.DataFrame,
) -> pd.DataFrame:
    """
    Compara en puntos:
      - modelo_delta: y_pred ya está en puntos
      - modelo_ret: convierte y_pred (retorno) a puntos usando price_t
    Requiere que ambas tablas estén alineadas por índice y split.
    """
    rows = []

    for sp in ["valid", "test"]:
        d = pred_delta[pred_delta["split"] == sp].copy()
        r = pred_ret[pred_ret["split"] == sp].copy()

        # Alinear timestamps (por seguridad)
        common_idx = d.index.intersection(r.index)
        d = d.loc[common_idx]
        r = r.loc[common_idx]

        y_true_pts = d["y_true"].to_numpy()          # delta real (puntos)
        y_pred_delta_pts = d["y_pred"].to_numpy()    # pred delta (puntos)
        y_pred_ret_pts = returns_to_points(
            r["y_pred"].to_numpy(),                  # ret predicho
            r["price_t"].to_numpy(),                 # close_t
        )

        m_delta = compute_point_metrics(y_true_pts, y_pred_delta_pts)
        m_ret   = compute_point_metrics(y_true_pts, y_pred_ret_pts)

        rows.append({
            "split": sp,
            "horizon": 60,
            "model": "ridge_delta",
            **m_delta
        })
        rows.append({
            "split": sp,
            "horizon": 60,
            "model": "ridge_ret_to_pts",
            **m_ret
        })

    return pd.DataFrame(rows)




,split,horizon,model,MAE_pts,RMSE_pts
0,valid,60,ridge_delta,6.344638,9.216015
1,valid,60,ridge_ret_to_pts,6.345231,9.216247
2,test,60,ridge_delta,9.723379,15.238656
3,test,60,ridge_ret_to_pts,9.724685,15.250917


In [74]:
# Predicciones Ridge AR(1)
pred_delta_60 = ridge_ar_predictions(
    mnq_intraday_targets,
    target_col="delta_60",
    split_s=split_s,
    alpha=1.0,
)

pred_ret_60 = ridge_ar_predictions(
    mnq_intraday_targets,
    target_col="ret_60",
    split_s=split_s,
    alpha=1.0,
)

cmp_pts_60 = compare_in_points(pred_delta_60, pred_ret_60)
cmp_pts_60["horizon"] = 60


In [73]:
pred_delta_90 = ridge_ar_predictions(
    mnq_intraday_targets,
    target_col="delta_90",
    split_s=split_s,
    alpha=1.0,
)

pred_ret_90 = ridge_ar_predictions(
    mnq_intraday_targets,
    target_col="ret_90",
    split_s=split_s,
    alpha=1.0,
)

cmp_pts_90 = compare_in_points(pred_delta_90, pred_ret_90)
cmp_pts_90["horizon"] = 90


In [75]:
display(cmp_pts_60)
display(cmp_pts_90)

,split,horizon,model,MAE_pts,RMSE_pts
0,valid,60,ridge_delta,6.344638,9.216015
1,valid,60,ridge_ret_to_pts,6.345231,9.216247
2,test,60,ridge_delta,9.723379,15.238656
3,test,60,ridge_ret_to_pts,9.724685,15.250917


,split,horizon,model,MAE_pts,RMSE_pts
0,valid,90,ridge_delta,6.533690,9.344644
1,valid,90,ridge_ret_to_pts,6.534570,9.345832
2,test,90,ridge_delta,9.966572,15.441234
3,test,90,ridge_ret_to_pts,9.968795,15.455628


#### 3.3.3.d Interpretación correcta


**Resultados – Evaluación doble en puntos (Paso 3.3.3)**

Se comparó el desempeño económico en puntos de:

- **Ridge entrenado directamente sobre deltas en puntos**
- **Ridge entrenado sobre retornos, con conversión posterior a puntos**  
  \(\widehat{\Delta P}_{t,h} = P_t \cdot \widehat{r}_{t,h}\)

La evaluación se realizó en los mismos splits temporales (valid / test) y para ambos horizontes.

---

**Horizonte H = 60 minutos**

- En **valid**, los errores en puntos son prácticamente idénticos:
  - MAE ≈ 6.34 pts
  - RMSE ≈ 9.22 pts
- En **test**, la diferencia sigue siendo mínima:
  - MAE ≈ 9.72 pts
  - RMSE ≈ 15.24–15.25 pts

No se observa penalización económica al formular el modelo en retornos.

---

**Horizonte H = 90 minutos**

- En **valid**, ambos enfoques producen errores casi indistinguibles:
  - MAE ≈ 6.53 pts
  - RMSE ≈ 9.35 pts
- En **test**, las diferencias siguen siendo marginales:
  - MAE ≈ 9.97 pts
  - RMSE ≈ 15.44–15.46 pts

El comportamiento se mantiene consistente al aumentar el horizonte.

---

**Interpretación**

- La conversión de retornos a puntos **no introduce degradación económica relevante**.
- El modelo entrenado en retornos es capaz de alcanzar **el mismo nivel de precisión en puntos** que el modelo entrenado directamente en deltas.
- Esto indica que la información predictiva aprendida en el espacio relativo se preserva al pasar al espacio económico absoluto.

---

**Conclusión del punto 3.3.3**

La evaluación doble confirma que:

- **delta y retorno son equivalentes desde el punto de vista económico**, cuando se comparan en puntos,
- la formulación en retornos **no sacrifica desempeño operativo**,
- y permite mantener una comparación justa entre enfoques.

Este resultado es clave porque desacopla la decisión del target de la métrica económica:  
la elección entre delta y retorno puede basarse en **criterios de estabilidad y generalización**, sin perder eficiencia en términos de puntos.


### 3.3.4. Cierre de la sección 3.3 – Evaluación preliminar de predecibilidad

En esta sección se evaluó la predecibilidad del target antes de introducir modelos complejos, siguiendo un enfoque incremental y controlado:

1. **Baselines Naive (zero y mean)**  
   Mostraron que existe, como máximo, una señal promedio muy débil, insuficiente por sí sola para justificar un modelo predictivo útil, pero consistente entre splits.

2. **Modelo lineal autoregresivo (Ridge AR(1))**  
   Un modelo lineal extremadamente simple fue capaz de capturar una fuerte dependencia temporal en el target, con resultados altamente estables entre valid y test, tanto para deltas como para retornos.

3. **Evaluación doble (escala del target y puntos)**  
   Al convertir las predicciones de retornos a puntos y evaluarlas en el mismo espacio económico que los deltas, se observó que:
   - el error en puntos es prácticamente idéntico,
   - no existe penalización económica por formular el modelo en retornos,
   - la información predictiva se preserva completamente tras la conversión.

Estos resultados confirman que **ambas formulaciones son igualmente predecibles y económicamente equivalentes** cuando se evalúan de forma justa.

# **4. Definición del target de predicción**

Integrando los resultados de las secciones **3.1 (definición empírica)**, **3.2 (estabilidad temporal)** y **3.3 (predecibilidad)**, se define el target de predicción principal como:

$$
r_{t,h} = \frac{P_{t+h} - P_t}{P_t}
$$

donde:
- $ P_t $ es el precio de cierre en el instante $t$,
- $ h $ es el horizonte de predicción (60 o 90 minutos).

**Justificación de la elección**

La elección del **retorno** como target se fundamenta en que:

- presenta **mayor estabilidad intradía e interdiaria**,
- es **más invariante al nivel de precio**, favoreciendo la generalización temporal,
- mantiene **idéntico desempeño económico** al delta cuando se evalúa en puntos,
- desacopla el aprendizaje estadístico de la escala absoluta del precio.

El **delta en puntos** no se descarta conceptualmente, sino que queda relegado al plano operativo:
- como métrica económica,
- como variable de evaluación,
- y como unidad natural de PnL.

---

**Implicancia para el resto del proyecto**

A partir de este punto:
- **todos los modelos predictivos se entrenarán sobre retornos**,
- las evaluaciones económicas se realizarán **en puntos** mediante conversión,
- y los criterios de éxito se definirán en términos de **error económico y estabilidad**, no solo métricas estadísticas.

Con esta definición, queda formalmente cerrada la etapa de investigación del target y se habilita el paso al **modelado definitivo**.
